In [558]:

# SEAG qualification period is 22 Oct 2024 to 5 Sept 2025

#1. Segregate into Male and Femal 
#2. For each gender perform the following: 
#a. Sort data by mapped eent, then perf scalar (higher the better)
#b. Identify tiers based on performance - Tier 1 is meets bronze medal mark for SEAG, Tier 2 is 2% and Tier 3 is 3.5%
#c. Check - if athlete met bronze or 2%/3.5% then delta_benchmark is zero or +, delta2% is + and delta 3.5% is +
#d. Top ranked athletes for each event are chosen. Max number of athletes for each event is 3, except for 100m/400m which is 6
#    This includes athletes on spex scholarship and potential
#e. The max for each tier is 2. Lower ranked athletes move down one tier.
#3. If athlete qualifies for more than one event the higher tier event is given
#4. Jump and throws junior program to be solved separately

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [559]:
# Import usual modules
import pandas as pd
import csv
import math
import os
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import openpyxl
import datetime
from scipy.stats import lognorm
import re
import string
from bs4 import BeautifulSoup
import requests
import unicodedata # for removing accented characters
import datetime
import icecream as ic
import dateutil.parser as parser 
import datacompy
import pytz

from itertools import permutations



from google.cloud import storage



In [560]:
'''
# PRODUCTION ENVIRONMENT
# Extract timed event records

import pandas_gbq
from google.oauth2 import service_account

credentials = service_account.Credentials.from_service_account_file(
    '/Users/veesheenyuen/Desktop/DataScience/Keys/saa-analytics-7c8937b70609.json',
    
    
)

sql1="""
SELECT NAME, RESULT, TEAM, AGE, RANK AS COMPETITION_RANK, DIVISION, EVENT, DISTANCE, EVENT_CLASS, UNIQUE_ID, DOB, NATIONALITY, WIND, CATEGORY_EVENT, GENDER, COMPETITION, DATE, YEAR, REGION, TIMESTAMP
FROM `saa-analytics.results.athlete_results_prod` 
WHERE RESULT!='NM' AND RESULT!='-' AND RESULT!='DNS' AND RESULT!='DNF' AND RESULT!='DNQ' AND RESULT!='DQ' AND RESULT IS NOT NULL

"""

competitors = pandas_gbq.read_gbq(sql1, project_id="saa-analytics", credentials=credentials)


'''

'\n# PRODUCTION ENVIRONMENT\n# Extract timed event records\n\nimport pandas_gbq\nfrom google.oauth2 import service_account\n\ncredentials = service_account.Credentials.from_service_account_file(\n    \'/Users/veesheenyuen/Desktop/DataScience/Keys/saa-analytics-7c8937b70609.json\',\n    \n    \n)\n\nsql1="""\nSELECT NAME, RESULT, TEAM, AGE, RANK AS COMPETITION_RANK, DIVISION, EVENT, DISTANCE, EVENT_CLASS, UNIQUE_ID, DOB, NATIONALITY, WIND, CATEGORY_EVENT, GENDER, COMPETITION, DATE, YEAR, REGION, TIMESTAMP\nFROM `saa-analytics.results.athlete_results_prod` \nWHERE RESULT!=\'NM\' AND RESULT!=\'-\' AND RESULT!=\'DNS\' AND RESULT!=\'DNF\' AND RESULT!=\'DNQ\' AND RESULT!=\'DQ\' AND RESULT IS NOT NULL\n\n"""\n\ncompetitors = pandas_gbq.read_gbq(sql1, project_id="saa-analytics", credentials=credentials)\n\n\n'

In [561]:
# PRODUCTION ENVIRONMENT
# Extract timed event records

import pandas_gbq
from google.oauth2 import service_account

credentials = service_account.Credentials.from_service_account_file(
    '/Users/veesheenyuen/Desktop/DataScience/Keys/saa-analytics-7c8937b70609.json',
    
    
)

sql1="""
SELECT NAME, RESULT, TEAM, AGE, RANK AS COMPETITION_RANK, DIVISION, EVENT, DISTANCE, EVENT_CLASS, UNIQUE_ID, DOB, NATIONALITY, WIND, CATEGORY_EVENT, GENDER, COMPETITION, DATE, YEAR, REGION, TIMESTAMP
FROM `saa-analytics.results.PRODUCTION` 
WHERE RESULT!='NM' AND RESULT!='-' AND RESULT!='DNS' AND RESULT!='DNF' AND RESULT!='DNQ' AND RESULT!='DQ' AND RESULT IS NOT NULL

"""

competitors = pandas_gbq.read_gbq(sql1, project_id="saa-analytics", credentials=credentials)



Downloading: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████|


In [562]:
competitors

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,DOB,NATIONALITY,WIND,CATEGORY_EVENT,GENDER,COMPETITION,DATE,YEAR,REGION,TIMESTAMP
0,NG Caitlin Shan Wen,12.42,<NA>,<NA>,5,<NA>,100m,<NA>,nan,<NA>,2010-11-24 00:00:00+00:00,SGP,-0.3,Sprint,Female,3rd Asian Youth Games,2025-10-23 00:00:00+00:00,2025,International,2025-10-27 21:31:00+00:00
1,TAN Ying Tong Shannon,12.53,<NA>,<NA>,2,<NA>,100m,<NA>,nan,<NA>,2009-11-20 00:00:00+00:00,SGP,-0.7,Sprint,Female,3rd Asian Youth Games,2025-10-23 00:00:00+00:00,2025,International,2025-10-27 21:31:00+00:00
2,NG Caitlin Shan Wen,12.36,<NA>,<NA>,6,<NA>,100m,<NA>,nan,<NA>,2010-11-24 00:00:00+00:00,SGP,0.4,Sprint,Female,3rd Asian Youth Games,2025-10-23 00:00:00+00:00,2025,International,2025-10-27 21:31:00+00:00
3,TAN Ying Tong Shannon,12.33,<NA>,<NA>,7,<NA>,100m,<NA>,nan,<NA>,2009-11-20 00:00:00+00:00,SGP,0.8,Sprint,Female,3rd Asian Youth Games,2025-10-23 00:00:00+00:00,2025,International,2025-10-27 21:31:00+00:00
4,TAY Sean Russell,49.89,<NA>,<NA>,6,<NA>,400m,<NA>,nan,<NA>,2009-12-25 00:00:00+00:00,SGP,<NA>,Sprint,Male,3rd Asian Youth Games,2025-10-24 00:00:00+00:00,2025,International,2025-10-27 21:31:00+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
168782,"LAU, REINA",9.52,Singapore Sports School,16,4.0,,Triple Jump,,,,2007,,0,Jump,Female,Melacca Open,2023-11-19 00:00:00+00:00,2023,International,2025-08-10 17:19:00+00:00
168783,"LOW, MING DAO JASE",11.97,Singapore Sports School,15,7.0,,Triple Jump,,,,2008,,0,Jump,Male,Thailand Sports School Games,2023-07-29 00:00:00+00:00,2023,International,2025-08-10 17:19:00+00:00
168784,CHAN CHENG ERN JENNICA,9.83,Singapore Sports School,15,2.0,,Triple Jump,,,,2008,,0,Jump,Female,Melacca Open,2023-11-19 00:00:00+00:00,2023,International,2025-08-10 17:19:00+00:00
168785,Vanessa Lee,17:11.36,<NA>,<NA>,nan,<NA>,5000m,<NA>,<NA>,<NA>,<NA>,SGP,<NA>,Long,Female,4J Studios scottishathletics National Senior &...,2025-08-23 00:00:00+00:00,2025,International,2025-08-31 18:46:00+00:00


In [563]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/17th SEA Youth/')


competitors.to_csv('athletes_all_prod.csv', sep=',', encoding='utf-8-sig', index=False)

In [564]:
competitors

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,DOB,NATIONALITY,WIND,CATEGORY_EVENT,GENDER,COMPETITION,DATE,YEAR,REGION,TIMESTAMP
0,NG Caitlin Shan Wen,12.42,<NA>,<NA>,5,<NA>,100m,<NA>,nan,<NA>,2010-11-24 00:00:00+00:00,SGP,-0.3,Sprint,Female,3rd Asian Youth Games,2025-10-23 00:00:00+00:00,2025,International,2025-10-27 21:31:00+00:00
1,TAN Ying Tong Shannon,12.53,<NA>,<NA>,2,<NA>,100m,<NA>,nan,<NA>,2009-11-20 00:00:00+00:00,SGP,-0.7,Sprint,Female,3rd Asian Youth Games,2025-10-23 00:00:00+00:00,2025,International,2025-10-27 21:31:00+00:00
2,NG Caitlin Shan Wen,12.36,<NA>,<NA>,6,<NA>,100m,<NA>,nan,<NA>,2010-11-24 00:00:00+00:00,SGP,0.4,Sprint,Female,3rd Asian Youth Games,2025-10-23 00:00:00+00:00,2025,International,2025-10-27 21:31:00+00:00
3,TAN Ying Tong Shannon,12.33,<NA>,<NA>,7,<NA>,100m,<NA>,nan,<NA>,2009-11-20 00:00:00+00:00,SGP,0.8,Sprint,Female,3rd Asian Youth Games,2025-10-23 00:00:00+00:00,2025,International,2025-10-27 21:31:00+00:00
4,TAY Sean Russell,49.89,<NA>,<NA>,6,<NA>,400m,<NA>,nan,<NA>,2009-12-25 00:00:00+00:00,SGP,<NA>,Sprint,Male,3rd Asian Youth Games,2025-10-24 00:00:00+00:00,2025,International,2025-10-27 21:31:00+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
168782,"LAU, REINA",9.52,Singapore Sports School,16,4.0,,Triple Jump,,,,2007,,0,Jump,Female,Melacca Open,2023-11-19 00:00:00+00:00,2023,International,2025-08-10 17:19:00+00:00
168783,"LOW, MING DAO JASE",11.97,Singapore Sports School,15,7.0,,Triple Jump,,,,2008,,0,Jump,Male,Thailand Sports School Games,2023-07-29 00:00:00+00:00,2023,International,2025-08-10 17:19:00+00:00
168784,CHAN CHENG ERN JENNICA,9.83,Singapore Sports School,15,2.0,,Triple Jump,,,,2008,,0,Jump,Female,Melacca Open,2023-11-19 00:00:00+00:00,2023,International,2025-08-10 17:19:00+00:00
168785,Vanessa Lee,17:11.36,<NA>,<NA>,nan,<NA>,5000m,<NA>,<NA>,<NA>,<NA>,SGP,<NA>,Long,Female,4J Studios scottishathletics National Senior &...,2025-08-23 00:00:00+00:00,2025,International,2025-08-31 18:46:00+00:00


In [565]:
competitors[competitors['NAME']=='MOHAMED IMRAN Ahmad Zubayr Bin']

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,DOB,NATIONALITY,WIND,CATEGORY_EVENT,GENDER,COMPETITION,DATE,YEAR,REGION,TIMESTAMP
10,MOHAMED IMRAN Ahmad Zubayr Bin,06:28.37,<NA>,<NA>,8,<NA>,2000m Steeplechase,<NA>,nan,<NA>,2009-03-17 00:00:00+00:00,SGP,<NA>,Steeple,Male,3rd Asian Youth Games,2025-10-25 00:00:00+00:00,2025,International,2025-10-27 21:31:00+00:00


In [566]:
#assert competitors['event_date'].all()!='nan' # make sure the conversion to datetime format is valid
#assert str(competitors['event_date'].all())!='NaT' # make sure the conversion to datetime format is valid


In [567]:
'''

# Calculate number of days from today to event date

competitors['event_date_dt'] = pd.to_datetime(competitors['event_date'], format='mixed', dayfirst=False)

competitors['delta_time']= datetime.datetime.now() - competitors['event_date_dt']

competitors['delta_time_conv'] = pd.to_numeric(competitors['delta_time'].dt.days, downcast='integer')

competitors['event_month'] = competitors['event_date_dt'].dt.month
'''

"\n\n# Calculate number of days from today to event date\n\ncompetitors['event_date_dt'] = pd.to_datetime(competitors['event_date'], format='mixed', dayfirst=False)\n\ncompetitors['delta_time']= datetime.datetime.now() - competitors['event_date_dt']\n\ncompetitors['delta_time_conv'] = pd.to_numeric(competitors['delta_time'].dt.days, downcast='integer')\n\ncompetitors['event_month'] = competitors['event_date_dt'].dt.month\n"

In [568]:
# DATE column to contain timezone - tz aware mode

competitors['DATE'] = pd.to_datetime(competitors['DATE'], format='mixed', dayfirst=False, utc=True)


In [569]:
# datetime to contain UTC (timezone)

competitors['NOW'] = datetime.datetime.now()

timezone = pytz.timezone('UTC')

competitors['NOW'] = datetime.datetime.now().replace(tzinfo=timezone)

In [570]:
competitors['NOW']

0        2025-10-29 10:18:18.246231+00:00
1        2025-10-29 10:18:18.246231+00:00
2        2025-10-29 10:18:18.246231+00:00
3        2025-10-29 10:18:18.246231+00:00
4        2025-10-29 10:18:18.246231+00:00
                       ...               
168782   2025-10-29 10:18:18.246231+00:00
168783   2025-10-29 10:18:18.246231+00:00
168784   2025-10-29 10:18:18.246231+00:00
168785   2025-10-29 10:18:18.246231+00:00
168786   2025-10-29 10:18:18.246231+00:00
Name: NOW, Length: 168787, dtype: datetime64[us, UTC]

In [571]:
# Calculate number of days from today to event date

#competitors['DATE'] = pd.to_datetime(competitors['DATE'], format='mixed', dayfirst=False, utc=False)

competitors['delta_time'] = competitors['NOW'] - competitors['DATE']


#competitors['delta_time'] = datetime.datetime.now() - competitors['DATE']


competitors['delta_time_conv'] = pd.to_numeric(competitors['delta_time'].dt.days, downcast='integer')

competitors['event_month'] = competitors['DATE'].dt.month


In [572]:
competitors['delta_time']

0          6 days 10:18:18.246231
1          6 days 10:18:18.246231
2          6 days 10:18:18.246231
3          6 days 10:18:18.246231
4          5 days 10:18:18.246231
                   ...           
168782   710 days 10:18:18.246231
168783   823 days 10:18:18.246231
168784   710 days 10:18:18.246231
168785    67 days 10:18:18.246231
168786    67 days 10:18:18.246231
Name: delta_time, Length: 168787, dtype: timedelta64[ns]

In [573]:
# Make sure date conversion is is valid for all rows

assert not competitors['delta_time'].isna().any()

In [574]:
# These results have not had their event dates converted properly

competitors[competitors['DATE'].isna()]

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,GENDER,COMPETITION,DATE,YEAR,REGION,TIMESTAMP,NOW,delta_time,delta_time_conv,event_month


In [575]:
competitors

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,GENDER,COMPETITION,DATE,YEAR,REGION,TIMESTAMP,NOW,delta_time,delta_time_conv,event_month
0,NG Caitlin Shan Wen,12.42,<NA>,<NA>,5,<NA>,100m,<NA>,nan,<NA>,...,Female,3rd Asian Youth Games,2025-10-23 00:00:00+00:00,2025,International,2025-10-27 21:31:00+00:00,2025-10-29 10:18:18.246231+00:00,6 days 10:18:18.246231,6,10
1,TAN Ying Tong Shannon,12.53,<NA>,<NA>,2,<NA>,100m,<NA>,nan,<NA>,...,Female,3rd Asian Youth Games,2025-10-23 00:00:00+00:00,2025,International,2025-10-27 21:31:00+00:00,2025-10-29 10:18:18.246231+00:00,6 days 10:18:18.246231,6,10
2,NG Caitlin Shan Wen,12.36,<NA>,<NA>,6,<NA>,100m,<NA>,nan,<NA>,...,Female,3rd Asian Youth Games,2025-10-23 00:00:00+00:00,2025,International,2025-10-27 21:31:00+00:00,2025-10-29 10:18:18.246231+00:00,6 days 10:18:18.246231,6,10
3,TAN Ying Tong Shannon,12.33,<NA>,<NA>,7,<NA>,100m,<NA>,nan,<NA>,...,Female,3rd Asian Youth Games,2025-10-23 00:00:00+00:00,2025,International,2025-10-27 21:31:00+00:00,2025-10-29 10:18:18.246231+00:00,6 days 10:18:18.246231,6,10
4,TAY Sean Russell,49.89,<NA>,<NA>,6,<NA>,400m,<NA>,nan,<NA>,...,Male,3rd Asian Youth Games,2025-10-24 00:00:00+00:00,2025,International,2025-10-27 21:31:00+00:00,2025-10-29 10:18:18.246231+00:00,5 days 10:18:18.246231,5,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
168782,"LAU, REINA",9.52,Singapore Sports School,16,4.0,,Triple Jump,,,,...,Female,Melacca Open,2023-11-19 00:00:00+00:00,2023,International,2025-08-10 17:19:00+00:00,2025-10-29 10:18:18.246231+00:00,710 days 10:18:18.246231,710,11
168783,"LOW, MING DAO JASE",11.97,Singapore Sports School,15,7.0,,Triple Jump,,,,...,Male,Thailand Sports School Games,2023-07-29 00:00:00+00:00,2023,International,2025-08-10 17:19:00+00:00,2025-10-29 10:18:18.246231+00:00,823 days 10:18:18.246231,823,7
168784,CHAN CHENG ERN JENNICA,9.83,Singapore Sports School,15,2.0,,Triple Jump,,,,...,Female,Melacca Open,2023-11-19 00:00:00+00:00,2023,International,2025-08-10 17:19:00+00:00,2025-10-29 10:18:18.246231+00:00,710 days 10:18:18.246231,710,11
168785,Vanessa Lee,17:11.36,<NA>,<NA>,nan,<NA>,5000m,<NA>,<NA>,<NA>,...,Female,4J Studios scottishathletics National Senior &...,2025-08-23 00:00:00+00:00,2025,International,2025-08-31 18:46:00+00:00,2025-10-29 10:18:18.246231+00:00,67 days 10:18:18.246231,67,8


In [ ]:
# SelCompeition Qualification Period

In [576]:
# Choose date range for SEAG qualification window from Oct 22 to current


competitors['DATE']=competitors['DATE'].dt.tz_localize(None)  # switch off timezone for compatibility with np.datetime64

start = datetime.datetime(2024, 10, 26)
#start = datetime.datetime(2025, 10, 26)


end = datetime.datetime(2025, 10, 26)

start_date = np.datetime64(start)
end_date = np.datetime64(end)


mask = (competitors['DATE'] >= start_date) & (competitors['DATE'] <= end_date)
athletes_selected = competitors.loc[mask]



In [577]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/17th SEA Youth/')

athletes_selected.to_csv('athletes_selected_oct29.csv', encoding='utf-8')

In [578]:
# Select all of 2024/25 for OCTC

#athletes_selected = competitors[(competitors['YEAR']=='2024')|(competitors['YEAR']=='2025')]

#athletes_selected = competitors[(competitors['YEAR']=='2025')]

In [579]:
athletes_selected

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,GENDER,COMPETITION,DATE,YEAR,REGION,TIMESTAMP,NOW,delta_time,delta_time_conv,event_month
0,NG Caitlin Shan Wen,12.42,<NA>,<NA>,5,<NA>,100m,<NA>,nan,<NA>,...,Female,3rd Asian Youth Games,2025-10-23,2025,International,2025-10-27 21:31:00+00:00,2025-10-29 10:18:18.246231+00:00,6 days 10:18:18.246231,6,10
1,TAN Ying Tong Shannon,12.53,<NA>,<NA>,2,<NA>,100m,<NA>,nan,<NA>,...,Female,3rd Asian Youth Games,2025-10-23,2025,International,2025-10-27 21:31:00+00:00,2025-10-29 10:18:18.246231+00:00,6 days 10:18:18.246231,6,10
2,NG Caitlin Shan Wen,12.36,<NA>,<NA>,6,<NA>,100m,<NA>,nan,<NA>,...,Female,3rd Asian Youth Games,2025-10-23,2025,International,2025-10-27 21:31:00+00:00,2025-10-29 10:18:18.246231+00:00,6 days 10:18:18.246231,6,10
3,TAN Ying Tong Shannon,12.33,<NA>,<NA>,7,<NA>,100m,<NA>,nan,<NA>,...,Female,3rd Asian Youth Games,2025-10-23,2025,International,2025-10-27 21:31:00+00:00,2025-10-29 10:18:18.246231+00:00,6 days 10:18:18.246231,6,10
4,TAY Sean Russell,49.89,<NA>,<NA>,6,<NA>,400m,<NA>,nan,<NA>,...,Male,3rd Asian Youth Games,2025-10-24,2025,International,2025-10-27 21:31:00+00:00,2025-10-29 10:18:18.246231+00:00,5 days 10:18:18.246231,5,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
167900,Marc Brian Louis,20.89,,,1,,200m,,,,...,Male,2nd SANO Sprint,2025-08-11,2025,International,2025-08-14 12:50:00+00:00,2025-10-29 10:18:18.246231+00:00,79 days 10:18:18.246231,79,8
167922,Harry Irfan Curra,34.57,,,3,,300m,,,,...,Male,Summer Games In Hikone,2025-08-09,2025,International,2025-08-14 12:50:00+00:00,2025-10-29 10:18:18.246231+00:00,81 days 10:18:18.246231,81,8
168210,Rui Yong Soh,14:46.81,,,4,,5000m,,,,...,Male,Hercules Wimbledon 5000m Festival Evening,2025-08-09,2025,International,2025-08-14 12:50:00+00:00,2025-10-29 10:18:18.246231+00:00,81 days 10:18:18.246231,81,8
168785,Vanessa Lee,17:11.36,<NA>,<NA>,nan,<NA>,5000m,<NA>,<NA>,<NA>,...,Female,4J Studios scottishathletics National Senior &...,2025-08-23,2025,International,2025-08-31 18:46:00+00:00,2025-10-29 10:18:18.246231+00:00,67 days 10:18:18.246231,67,8


In [580]:
athletes_selected[athletes_selected['NAME']=='MOHAMED IMRAN Ahmad Zubayr Bin']

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,GENDER,COMPETITION,DATE,YEAR,REGION,TIMESTAMP,NOW,delta_time,delta_time_conv,event_month
10,MOHAMED IMRAN Ahmad Zubayr Bin,06:28.37,<NA>,<NA>,8,<NA>,2000m Steeplechase,<NA>,nan,<NA>,...,Male,3rd Asian Youth Games,2025-10-25,2025,International,2025-10-27 21:31:00+00:00,2025-10-29 10:18:18.246231+00:00,4 days 10:18:18.246231,4,10


In [581]:
# Choose 2024/25 only

athletes = athletes_selected

In [582]:
athletes

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,GENDER,COMPETITION,DATE,YEAR,REGION,TIMESTAMP,NOW,delta_time,delta_time_conv,event_month
0,NG Caitlin Shan Wen,12.42,<NA>,<NA>,5,<NA>,100m,<NA>,nan,<NA>,...,Female,3rd Asian Youth Games,2025-10-23,2025,International,2025-10-27 21:31:00+00:00,2025-10-29 10:18:18.246231+00:00,6 days 10:18:18.246231,6,10
1,TAN Ying Tong Shannon,12.53,<NA>,<NA>,2,<NA>,100m,<NA>,nan,<NA>,...,Female,3rd Asian Youth Games,2025-10-23,2025,International,2025-10-27 21:31:00+00:00,2025-10-29 10:18:18.246231+00:00,6 days 10:18:18.246231,6,10
2,NG Caitlin Shan Wen,12.36,<NA>,<NA>,6,<NA>,100m,<NA>,nan,<NA>,...,Female,3rd Asian Youth Games,2025-10-23,2025,International,2025-10-27 21:31:00+00:00,2025-10-29 10:18:18.246231+00:00,6 days 10:18:18.246231,6,10
3,TAN Ying Tong Shannon,12.33,<NA>,<NA>,7,<NA>,100m,<NA>,nan,<NA>,...,Female,3rd Asian Youth Games,2025-10-23,2025,International,2025-10-27 21:31:00+00:00,2025-10-29 10:18:18.246231+00:00,6 days 10:18:18.246231,6,10
4,TAY Sean Russell,49.89,<NA>,<NA>,6,<NA>,400m,<NA>,nan,<NA>,...,Male,3rd Asian Youth Games,2025-10-24,2025,International,2025-10-27 21:31:00+00:00,2025-10-29 10:18:18.246231+00:00,5 days 10:18:18.246231,5,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
167900,Marc Brian Louis,20.89,,,1,,200m,,,,...,Male,2nd SANO Sprint,2025-08-11,2025,International,2025-08-14 12:50:00+00:00,2025-10-29 10:18:18.246231+00:00,79 days 10:18:18.246231,79,8
167922,Harry Irfan Curra,34.57,,,3,,300m,,,,...,Male,Summer Games In Hikone,2025-08-09,2025,International,2025-08-14 12:50:00+00:00,2025-10-29 10:18:18.246231+00:00,81 days 10:18:18.246231,81,8
168210,Rui Yong Soh,14:46.81,,,4,,5000m,,,,...,Male,Hercules Wimbledon 5000m Festival Evening,2025-08-09,2025,International,2025-08-14 12:50:00+00:00,2025-10-29 10:18:18.246231+00:00,81 days 10:18:18.246231,81,8
168785,Vanessa Lee,17:11.36,<NA>,<NA>,nan,<NA>,5000m,<NA>,<NA>,<NA>,...,Female,4J Studios scottishathletics National Senior &...,2025-08-23,2025,International,2025-08-31 18:46:00+00:00,2025-10-29 10:18:18.246231+00:00,67 days 10:18:18.246231,67,8


In [583]:
athletes[athletes['NAME']=='MOHAMED IMRAN Ahmad Zubayr Bin']

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,GENDER,COMPETITION,DATE,YEAR,REGION,TIMESTAMP,NOW,delta_time,delta_time_conv,event_month
10,MOHAMED IMRAN Ahmad Zubayr Bin,06:28.37,<NA>,<NA>,8,<NA>,2000m Steeplechase,<NA>,nan,<NA>,...,Male,3rd Asian Youth Games,2025-10-25,2025,International,2025-10-27 21:31:00+00:00,2025-10-29 10:18:18.246231+00:00,4 days 10:18:18.246231,4,10


In [584]:
# Process dates to extract age

# Map NSG divisions into age

mask = (athletes['DIVISION'].str.contains(r'A', na=False))
athletes.loc[mask, 'AGE'] = '18.5'

mask = (athletes['DIVISION'].str.contains(r'B', na=False))
athletes.loc[mask, 'AGE'] = '16'

mask = (athletes['DIVISION'].str.contains(r'C', na=False))
athletes.loc[mask, 'AGE'] = '13.5'

mask = (athletes['DIVISION'].str.contains(r'O', na=False))
athletes.loc[mask, 'AGE'] = '12'


In [585]:
def length(string):

    B = ''
    year = ''

    try:

        length = len(string)

        if length == 2:

            string = '19' + string

        elif length == 1:

            string = ''

        else:

            pass

        if string is not None or len(string) != 1:

            B = parser.parse(string, dayfirst=True)
                        
    except:

        pass

    return B


athletes['DOB_dt'] = athletes['DOB'].apply(length)




/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_16781/1704523743.py:33: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  athletes['DOB_dt'] = athletes['DOB'].apply(length)


In [586]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/17th SEA Youth/')


athletes.to_csv('U18_age_post_map.csv', sep=',', encoding='utf-8-sig', index=False)


In [587]:
athletes[athletes['NAME']=='MOHAMED IMRAN Ahmad Zubayr Bin']

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,COMPETITION,DATE,YEAR,REGION,TIMESTAMP,NOW,delta_time,delta_time_conv,event_month,DOB_dt
10,MOHAMED IMRAN Ahmad Zubayr Bin,06:28.37,<NA>,18.5,8,<NA>,2000m Steeplechase,<NA>,nan,<NA>,...,3rd Asian Youth Games,2025-10-25,2025,International,2025-10-27 21:31:00+00:00,2025-10-29 10:18:18.246231+00:00,4 days 10:18:18.246231,4,10,2009-03-17 00:00:00+00:00


# Map Events

In [588]:
# Create temporary mapped event column
# Junior Category Only

athletes['MAPPED_EVENT']=''

for col in athletes.columns:
    athletes[col] = athletes[col].astype(str)
    athletes[col] = athletes[col].str.replace('\xa0', ' ', regex=True)
    athletes[col] = athletes[col].str.replace('[\x00-\x1f\x7f-\x9f]', '', regex=True)
    athletes[col] = athletes[col].str.replace('\r', ' ', regex=True)
    athletes[col] = athletes[col].str.replace('\n', ' ', regex=True)
    athletes[col] = athletes[col].str.strip()


# Correct javelin category

mask = athletes['EVENT'].str.contains(r'Javelin', na=True)
athletes.loc[mask, 'CATEGORY_EVENT'] = 'Throw'


# Running

mask = (athletes['EVENT'].str.contains(r'Dash', na=True) & athletes['DISTANCE'].str.contains(r'100', na=True))
athletes.loc[mask, 'MAPPED_EVENT'] = '100m'
mask = (athletes['EVENT'].str.contains(r'Run', na=True) & athletes['DISTANCE'].str.contains(r'100', na=True))
athletes.loc[mask, 'MAPPED_EVENT'] = '100m'
mask = athletes['EVENT'].str.contains(r'100 Meter Run', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '100m'
mask = athletes['EVENT'].str.contains(r'^100m$', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '100m'

mask = (athletes['EVENT'].str.contains(r'Dash', na=True) & athletes['DISTANCE'].str.contains(r'200', na=True))
athletes.loc[mask, 'MAPPED_EVENT'] = '200m'
mask = (athletes['EVENT'].str.contains(r'Run', na=True) & athletes['DISTANCE'].str.contains(r'200', na=True))
athletes.loc[mask, 'MAPPED_EVENT'] = '200m'
mask = athletes['EVENT'].str.contains(r'^200m$', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '200m'
mask = athletes['EVENT'].str.contains(r'200\sMeter', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '200m'

mask = (athletes['EVENT'].str.contains(r'Dash', na=True) & athletes['DISTANCE'].str.contains(r'400', na=True))
athletes.loc[mask, 'MAPPED_EVENT'] = '400m'
mask = athletes['EVENT'].str.contains(r'^400m$', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '400m'
mask = athletes['EVENT'].str.contains(r'^400\sMeter$', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '400m'
mask = (athletes['EVENT'].str.contains(r'Run', na=True) & athletes['DISTANCE'].str.contains(r'400', na=True))
athletes.loc[mask, 'MAPPED_EVENT'] = '400m'


mask = (athletes['EVENT'].str.contains(r'Run', na=True) & athletes['DISTANCE'].str.contains(r'800', na=True))
athletes.loc[mask, 'MAPPED_EVENT'] = '800m'
mask = athletes['EVENT'].str.contains(r'800 Meter Run', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '800m'
mask = athletes['EVENT'].str.contains(r'^800m$', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '800m'
mask = (athletes['EVENT'].str.contains(r'Run', na=True) & athletes['DISTANCE'].str.contains(r'1000', na=True))
athletes.loc[mask, 'MAPPED_EVENT'] = '1000m'


mask = (athletes['EVENT'].str.contains(r'Run', na=True) & athletes['DISTANCE'].str.contains(r'1500', na=True))
athletes.loc[mask, 'MAPPED_EVENT'] = '1500m'
mask = athletes['EVENT'].str.contains(r'^1500m$', na=True, regex=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '1500m'
mask = (athletes['EVENT'].str.contains(r'Run', na=True) & athletes['DISTANCE'].str.contains(r'3000', na=True))
athletes.loc[mask, 'MAPPED_EVENT'] = '3000m'
#mask = athletes['EVENT'].str.contains(r'3000m', na=True)
#athletes.loc[mask, 'MAPPED_EVENT'] = '3000m'
mask = (athletes['EVENT'].str.contains(r'Run', na=True) & athletes['DISTANCE'].str.contains(r'5000', na=True))
athletes.loc[mask, 'MAPPED_EVENT'] = '5000m'
mask = athletes['EVENT'].str.contains(r'^5000m$', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '5000m'
mask = (athletes['EVENT'].str.contains(r'Run', na=True) & athletes['DISTANCE'].str.contains(r'10000', na=True))
athletes.loc[mask, 'MAPPED_EVENT'] = '10,000m'
mask = athletes['EVENT'].str.contains(r'^10000m$', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '10,000m'
mask = athletes['EVENT'].str.contains(r'^10\,000m$', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '10,000m'
mask = (athletes['EVENT'].str.contains(r'Run', na=True) & athletes['DISTANCE'].str.contains(r'Mile', na=True))
athletes.loc[mask, 'MAPPED_EVENT'] = '1 Mile'

#mask = athletes['EVENT'].str.contains(r'10\,000m', na=True, regex=True)
#athletes.loc[mask, 'MAPPED_EVENT'] = '10000m'



# Hurdles

#mask = athletes['EVENT'].str.contains(r'100\sMeter\sHurdles\s\(0\.838m\)', na=True)
#athletes.loc[mask, 'MAPPED_EVENT'] = '100m Hurdles'
#mask = athletes['EVENT'].str.contains(r'100m\sHurdles\s\(0\.838m\)', na=True)
#athletes.loc[mask, 'MAPPED_EVENT'] = '100m Hurdles'


mask = athletes['EVENT'].str.contains(r'110m\sHurdles\s\(0\.914m\)', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '110m hurdles'
##mask = athletes['EVENT'].str.contains(r'110m\sHurdles\s\(1\.067m\)', na=True)
##athletes.loc[mask, 'MAPPED_EVENT'] = '110m Hurdles'


mask = (athletes['EVENT'].str.contains(r'100m Hurdles|100m hurdles', na=False) & athletes['EVENT_CLASS'].str.contains('0.762', na=False) & athletes['GENDER'].str.contains(r'Female', na=False))  # this is the correct syntax
athletes.loc[mask, 'MAPPED_EVENT'] = '100m Hurdles'
#mask = (athletes['EVENT'].str.contains(r'100m Hurdles|100m hurdles', na=False) & athletes['DIVISION'].str.contains('None', na=False) & athletes['GENDER'].str.contains(r'Female', na=False) & athletes['REGION'].str.contains(r'International', na=False))  # this is the correct syntax
#athletes.loc[mask, 'MAPPED_EVENT'] = '100m Hurdles'
mask = (athletes['EVENT'].str.contains(r'^Hurdles$', na=False) & athletes['DISTANCE'].str.contains(r'100', na=False) & athletes['EVENT_CLASS'].str.contains(r'0.762', na=False) & athletes['GENDER'].str.contains(r'Female', na=False))
athletes.loc[mask, 'MAPPED_EVENT'] = '100m Hurdles'
#mask = (athletes['EVENT'].str.contains(r'100m Hurdles|100m hurdles', na=False) & athletes['REGION'].str.contains(r'International', na=False) & athletes['GENDER'].str.contains(r'Female', na=False))
#athletes.loc[mask, 'MAPPED_EVENT'] = '100m Hurdles'



#mask = (athletes['EVENT'].str.contains(r'^Hurdles$', na=False) & athletes['DISTANCE'].str.contains(r'110', na=False) & athletes['DIVISION'].str.contains(r'OPEN|Open', na=False) & athletes['GENDER'].str.contains(r'Male', na=False))
#athletes.loc[mask, 'MAPPED_EVENT'] = '110m Hurdles'
#mask = (athletes['EVENT'].str.contains(r'^Hurdles$', na=False) & athletes['DISTANCE'].str.contains(r'110', na=False) & athletes['EVENT_CLASS'].str.contains(r'0.838|0.84', na=False) & athletes['GENDER'].str.contains(r'Female', na=False))
#athletes.loc[mask, 'MAPPED_EVENT'] = '110m Hurdles'
#mask = ((athletes['EVENT'].str.contains(r'110m Hurdles|110m hurdles', na=False)) 
#         & ((athletes['EVENT_CLASS'].str.contains('None', na=False))|(athletes['EVENT_CLASS']==np.nan)|(athletes['EVENT_CLASS']=='')) 
#         & athletes['REGION'].str.contains(r'International', na=False) & (athletes['DIVISION'].str.contains(r'None', na=False)))  # this is the correct syntax
#athletes.loc[mask, 'MAPPED_EVENT'] = '110m Hurdles'

#mask = (athletes['EVENT'].str.contains(r'110m Hurdles|110m hurdles', na=False) & athletes['REGION'].str.contains(r'International', na=False) & athletes['GENDER'].str.contains(r'Male', na=False))
#athletes.loc[mask, 'MAPPED_EVENT'] = '110m Hurdles'


# Using np.where instead

#athletes['MAPPED_EVENT'] = np.where(((athletes['EVENT']=='110m hurdles|110m Hurdles') & ((athletes['EVENT_CLASS']=='')|athletes['EVENT_CLASS']=='None') & (athletes['REGION']=='International')), '110m Hurdles', ' ')   
                                
mask = (athletes['EVENT'].str.contains(r'110m Hurdles|110m hurdles', na=False) & athletes['EVENT_CLASS'].str.contains(r'0.914|91.4', na=False) & athletes['GENDER'].str.contains(r'Male', na=False))
athletes.loc[mask, 'MAPPED_EVENT'] = '110m Hurdles'


mask = (athletes['EVENT'].str.contains(r'^Hurdles$', na=False) & athletes['DISTANCE'].str.contains(r'110', na=False) & athletes['EVENT_CLASS'].str.contains(r'0.914', na=False))
athletes.loc[mask, 'MAPPED_EVENT'] = '110m Hurdles'
mask = (athletes['EVENT'].str.contains(r'^Hurdles$', na=False) & athletes['DISTANCE'].str.contains(r'110', na=False) & athletes['EVENT_CLASS'].str.contains(r'0.762', na=False) & athletes['GENDER'].str.contains(r'Female', na=False))
athletes.loc[mask, 'MAPPED_EVENT'] = '110m Hurdles'
mask = (athletes['EVENT'].str.contains(r'^Hurdles$', na=False) & athletes['DISTANCE'].str.contains(r'110', na=False) & athletes['DIVISION'].str.contains(r'U18', na=False))
athletes.loc[mask, 'MAPPED_EVENT'] = '110m Hurdles'

#mask = (athletes['EVENT'].str.contains(r'^Hurdles$', na=False) & athletes['DISTANCE'].str.contains(r'110', na=False) & athletes['EVENT_CLASS'].str.contains(' ', na=True))
#athletes.loc[mask, 'MAPPED_EVENT'] = '110m Hurdles'







#mask = (athletes['EVENT'].str.contains(r'^400m\sHurdles$', na=False) & athletes['EVENT_CLASS'].str.contains(r'', na=False))
#athletes.loc[mask, 'MAPPED_EVENT'] = '400m Hurdles'
#mask = athletes['EVENT'].str.contains(r'400m\sHurdles\s\(0.840m\)', na=True)
#athletes.loc[mask, 'MAPPED_EVENT'] = ' '
mask = (athletes['EVENT'].str.contains(r'^Hurdles$', na=False) & athletes['DISTANCE'].str.contains(r'400', na=False) & athletes['EVENT_CLASS'].str.contains(r'0.762|76.2cm', na=False) & athletes['GENDER'].str.contains(r'Female', na=False))
athletes.loc[mask, 'MAPPED_EVENT'] = '400m Hurdles'


#mask = athletes['EVENT'].str.contains(r'400\sMeter\sHurdles\s\(0\.914m\)', na=True)
#athletes.loc[mask, 'MAPPED_EVENT'] = '400m Hurdles'
#mask = athletes['EVENT'].str.contains(r'400m\sHurdles\s\(0\.914m\)', na=True)
#athletes.loc[mask, 'MAPPED_EVENT'] = '400m Hurdles'
mask = (athletes['EVENT'].str.contains(r'^Hurdles$', na=False) & athletes['DISTANCE'].str.contains(r'400', na=False) & athletes['EVENT_CLASS'].str.contains(r'0.84|84|0.838', na=False))
athletes.loc[mask, 'MAPPED_EVENT'] = '400m Hurdles'
mask = (athletes['EVENT'].str.contains(r'^Hurdles$', na=False) & athletes['DISTANCE'].str.contains(r'400', na=False) & athletes['DIVISION'].str.contains(r'U18', na=False))
athletes.loc[mask, 'MAPPED_EVENT'] = '400m Hurdles'

mask = (athletes['EVENT'].str.contains(r'400m Hurdles', na=False) & athletes['EVENT_CLASS'].str.contains(r'0.84|84|0.838', na=False)  & athletes['GENDER'].str.contains(r'Male', na=False))
athletes.loc[mask, 'MAPPED_EVENT'] = '400m Hurdles'


#mask = athletes['EVENT'].str.contains(r'^400\sMeter\sHurdles\s\(0\.762m\)$', na=True)
#athletes.loc[mask, 'MAPPED_EVENT'] = '400m Hurdles'
##mask = athletes['EVENT'].str.contains(r'^400m\sHurdles\s\(0\.762m\)$', na=True)
#athletes.loc[mask, 'MAPPED_EVENT'] = '400m Hurdles'
mask = (athletes['EVENT'].str.contains(r'Hurdles', na=False) & athletes['DISTANCE'].str.contains(r'400', na=False) & athletes['EVENT_CLASS'].str.contains(r'0.762', na=False)& athletes['GENDER'].str.contains(r'Female', na=False))
athletes.loc[mask, 'MAPPED_EVENT'] = '400m Hurdles'
mask = (athletes['EVENT'].str.contains(r'400m Hurdles', na=False) & athletes['EVENT_CLASS'].str.contains(r'0.762m', na=False) & athletes['GENDER'].str.contains(r'Female', na=False))
athletes.loc[mask, 'MAPPED_EVENT'] = '400m Hurdles'
mask = (athletes['EVENT'].str.contains(r'400m Hurdles|400m hurdles', na=False) & athletes['EVENT_CLASS'].str.contains('0.762|0.84|0.838', na=False) & athletes['REGION'].str.contains(r'International', na=False))  # this is the correct syntax
athletes.loc[mask, 'MAPPED_EVENT'] = '400m Hurdles'
mask = (athletes['EVENT'].str.contains(r'400m Hurdles|400m hurdles', na=False) & athletes['REGION'].str.contains(r'International', na=False))  # this is the correct syntax
athletes.loc[mask, 'MAPPED_EVENT'] = '400m Hurdles'



# Throws


#mask = ((athletes['EVENT'].str.contains(r'Javelin\sThrow\s\(600g\)', na=True, regex=True)) & (athletes['GENDER']=='Female'))
#athletes.loc[mask, 'MAPPED_EVENT'] = 'Javelin Throw'
#mask = ((athletes['EVENT'].str.contains(r'Javelin\sThrow\s600g', na=True, regex=True)) & (athletes['GENDER']=='Female'))
#athletes.loc[mask, 'MAPPED_EVENT'] = 'Javelin Throw'
#mask = ((athletes['EVENT'].str.contains(r'Javelin\sThrow\s600g\)', na=True, regex=True)) & (athletes['GENDER']=='Female'))
#athletes.loc[mask, 'MAPPED_EVENT'] = 'Javelin Throw'
#mask = athletes['EVENT'].str.contains(r'Javelin\sThrow\s\(800g\)', na=True, regex=True)
#athletes.loc[mask, 'MAPPED_EVENT'] = 'Javelin Throw'

mask = (athletes['EVENT'].str.contains(r'Javelin Throw|Javelin throw|Javelin', na=False) & athletes['EVENT_CLASS'].str.contains(r'500g', na=False) & athletes['GENDER'].str.contains(r'Female', na=False))
athletes.loc[mask, 'MAPPED_EVENT'] = 'Javelin Throw'
mask = (athletes['EVENT'].str.contains(r'Javelin Throw|Javelin throw|Javelin', na=False) & athletes['EVENT_CLASS'].str.contains(r'700g', na=False) & athletes['GENDER'].str.contains(r'Male', na=False))
athletes.loc[mask, 'MAPPED_EVENT'] = 'Javelin Throw'
#mask = (athletes['EVENT'].str.contains(r'Javelin Throw|Javelin throw', na=False) & athletes['DIVISION'].str.contains(r'OPEN|Open', na=False))
#athletes.loc[mask, 'MAPPED_EVENT'] = 'Javelin Throw'

mask = (athletes['EVENT'].str.contains(r'Shot Put|Shot put', na=False, regex=True) & (athletes['GENDER']=='Female') & (athletes['EVENT_CLASS']=='3kg'))# there are some additional characters after Put
athletes.loc[mask, 'MAPPED_EVENT'] = 'Shot Put'


#mask = athletes['EVENT'].str.contains(r'Women\sShot\sPut\s4kg\sOpen', na=True, regex=True)
#athletes.loc[mask, 'MAPPED_EVENT'] = 'Shot Put'
#mask = athletes['EVENT'].str.contains(r'Men\sShot\sPut\s4kg\sOPEN', na=True, regex=True)
#athletes.loc[mask, 'MAPPED_EVENT'] = 'Shot put'


#mask = athletes['EVENT'].str.contains(r'Women\sShot\sPut\s\(4kg\)', na=True, regex=True)
#athletes.loc[mask, 'MAPPED_EVENT'] = 'Shot Put'
#mask = athletes['EVENT'].str.contains(r'Shot\sPut\s\(7\.26kg\)', na=True, regex=True)
#athletes.loc[mask, 'MAPPED_EVENT'] = 'Shot Put'
#mask = athletes['EVENT'].str.contains(r'Shot\sPut\s7\.26kg\sOpen', na=True, regex=True)
#athletes.loc[mask, 'MAPPED_EVENT'] = 'Shot Put'
mask = (athletes['EVENT'].str.contains(r'Shot Put|Shot put', na=False) & (athletes['GENDER']=='Male') & (athletes['EVENT_CLASS'].str.contains(r'5', na=False)))# there are some additional characters after Put
athletes.loc[mask, 'MAPPED_EVENT'] = 'Shot Put'
mask = (athletes['EVENT'].str.contains(r'Shot Put|Shot put', na=False) & (athletes['GENDER']=='Female') & (athletes['EVENT_CLASS'].str.contains(r'3', na=False)))# there are some additional characters after Put
athletes.loc[mask, 'MAPPED_EVENT'] = 'Shot Put'

#mask = (athletes['EVENT'].str.contains(r'Shot Put|Shot put', na=False) & (athletes['DIVISION'].str.contains(r'OPEN|Open', na=False)))# there are some additional characters after Put
#athletes.loc[mask, 'MAPPED_EVENT'] = 'Shot Put'

#mask = (athletes['EVENT'].str.contains(r'Shot Put|Shot put', na=False) & (athletes['REGION'].str.contains(r'International', na=False)) & athletes['EVENT_CLASS'].str.contains(r'None', na=False))# there are some additional characters after Put
#athletes.loc[mask, 'MAPPED_EVENT'] = 'Shot Put'



#mask = (athletes['EVENT'].str.contains(r'Hammer\sThrow\s\(4kg\)', na=True) & (athletes['GENDER']=='Female'))
#athletes.loc[mask, 'MAPPED_EVENT'] = 'Hammer Throw'
#mask = athletes['EVENT'].str.contains(r'Hammer\sThrow\s\(7\.26kg\)', na=True)
#athletes.loc[mask, 'MAPPED_EVENT'] = 'Hammer Throw'
mask = (athletes['EVENT'].str.contains(r'Hammer Throw|Hammer throw', na=False) & athletes['EVENT_CLASS'].str.contains(r'5kg', na=False))
athletes.loc[mask, 'MAPPED_EVENT'] = 'Hammer Throw'
mask = (athletes['EVENT'].str.contains(r'Hammer Throw|Hammer throw', na=False) & athletes['EVENT_CLASS'].str.contains(r'3.00kg', na=False))
athletes.loc[mask, 'MAPPED_EVENT'] = 'Hammer Throw'
#mask = (athletes['EVENT'].str.contains(r'Hammer Throw|Hammer throw', na=False) & (athletes['DIVISION'].str.contains(r'OPEN|Open', na=False)))# there are some additional characters after Put
#athletes.loc[mask, 'MAPPED_EVENT'] = 'Hammer Throw'



#mask = ((athletes['EVENT'].str.contains(r'Discus\sThrow\s\(1kg\)', na=False)) & (athletes['GENDER']=='Female'))
#athletes.loc[mask, 'MAPPED_EVENT'] = 'Discus Throw'

#mask = ((athletes['EVENT'].str.contains(r'Discus\s\(1\.00kg\)', na=False))  & (athletes['GENDER']=='Female'))
#athletes.loc[mask, 'MAPPED_EVENT'] = 'Discus Throw'


#mask = athletes['EVENT'].str.contains(r'Discus\sThrow\s\(2kg\)', na=False)
#athletes.loc[mask, 'MAPPED_EVENT'] = 'Discus Throw'
#mask = ((athletes['EVENT'].str.contains(r'Discus\sThrow\s\(1kg\)', na=False)) & (athletes['GENDER']=='Female'))
#athletes.loc[mask, 'MAPPED_EVENT'] = 'Discus Throw'
mask = (athletes['EVENT'].str.contains(r'Discus Throw|Discus|Discus throw', na=False) & athletes['EVENT_CLASS'].str.contains(r'1.5kg|1.50kg', na=False) & athletes['GENDER'].str.contains(r'Male', na=False))
athletes.loc[mask, 'MAPPED_EVENT'] = 'Discus Throw'
mask = (athletes['EVENT'].str.contains(r'Discus Throw|Discus|Discus throw', na=False) & athletes['EVENT_CLASS'].str.contains(r'1kg|1.00kg', na=False) & athletes['GENDER'].str.contains(r'Female', na=False))
athletes.loc[mask, 'MAPPED_EVENT'] = 'Discus Throw'

#mask = (athletes['EVENT'].str.contains(r'Discus Throw|Discus throw', na=False) & athletes['REGION'].str.contains(r'International', na=False))
#athletes.loc[mask, 'MAPPED_EVENT'] = 'Discus Throw'
mask = (athletes['EVENT'].str.contains(r'Discus Throw|Discus throw', na=False) & athletes['DIVISION'].str.contains(r'U18', na=False))
athletes.loc[mask, 'MAPPED_EVENT'] = 'Discus Throw'
#mask = (athletes['EVENT'].str.contains(r'Discus Throw|Discus throw', na=False) & athletes['DIVISION'].str.contains(r'None', na=False) & athletes['EVENT_CLASS'].str.contains(r'None', na=False))
#athletes.loc[mask, 'MAPPED_EVENT'] = 'Discus Throw'

#mask = (athletes['EVENT'].str.contains(r'Discus Throw|Discus throw', na=False) & athletes['REGION'].str.contains(r'International', na=False) & athletes['EVENT_CLASS'].str.contains(r'None', na=False))
#athletes.loc[mask, 'MAPPED_EVENT'] = 'Discus Throw'



# Jumps

mask = athletes['EVENT'].str.contains(r'High Jump', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'High Jump'

mask = athletes['EVENT'].str.contains(r'^Long\sJump$', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Long Jump'
mask = athletes['EVENT'].str.contains(r'Long Jump Open', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Long Jump'
mask = athletes['EVENT'].str.contains(r'Long Jump Trial', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Long Jump'


mask = athletes['EVENT'].str.contains(r'Triple Jump', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Triple Jump'
mask = athletes['EVENT'].str.contains(r'Pole Vault', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Pole Vault'
mask = athletes['EVENT'].str.contains(r'High jump', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'High Jump'
mask = athletes['EVENT'].str.contains(r'Long jump', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Long Jump'
mask = athletes['EVENT'].str.contains(r'Triple jump', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Triple Jump'
mask = athletes['EVENT'].str.contains(r'^Pole\svault$', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Pole Vault'

# Steeplechase

#mask = athletes['EVENT'].str.contains(r'2000m S/C', na=True)
#athletes.loc[mask, 'MAPPED_EVENT'] = '2000m Steeplechase'
#mask = athletes['EVENT'].str.contains(r'2000m steeplechase', na=True)
#athletes.loc[mask, 'MAPPED_EVENT'] = '2000m Steeplechase'
#mask = athletes['EVENT'].str.contains(r'2000 Meter Steeplechase', na=True)
#athletes.loc[mask, 'MAPPED_EVENT'] = '2000m Steeplechase'
#mask = (athletes['EVENT'].str.contains(r'2000m Steeplechase|2000m S\/C', na=True) & athletes['REGION'].str.contains(r'International', na=False))
#athletes.loc[mask, 'MAPPED_EVENT'] = '2000m Steeplechase'
mask = (athletes['EVENT'].str.contains(r'Steeplechase|steeplechase|S\/C', na=False) & athletes['DISTANCE'].str.contains(r'2000', na=False)  & athletes['EVENT_CLASS'].str.contains(r'0.914', na=False)  & athletes['GENDER'].str.contains(r'Male', na=False))
athletes.loc[mask, 'MAPPED_EVENT'] = '2000m Steeplechase'
mask = (athletes['EVENT'].str.contains(r'Steeplechase|steeplechase|S\/C', na=False) & athletes['DISTANCE'].str.contains(r'2000', na=False)  & athletes['EVENT_CLASS'].str.contains(r'0.762', na=False)  & athletes['GENDER'].str.contains(r'Female', na=False))
athletes.loc[mask, 'MAPPED_EVENT'] = '2000m Steeplechase'


mask = (athletes['EVENT'].str.contains(r'2000m Steeplechase|2000m steeplechase|2000m S\/C', na=False) & athletes['EVENT_CLASS'].str.contains(r'0.914', na=False) & athletes['GENDER'].str.contains(r'Male', na=False))
athletes.loc[mask, 'MAPPED_EVENT'] = '2000m Steeplechase'
mask = (athletes['EVENT'].str.contains(r'2000m Steeplechase|2000m steeplechase|2000m S\/C', na=False) & athletes['EVENT_CLASS'].str.contains(r'0.762', na=False) & athletes['GENDER'].str.contains(r'Female', na=False))
athletes.loc[mask, 'MAPPED_EVENT'] = '2000m Steeplechase'

mask = (athletes['EVENT'].str.contains(r'2000m Steeplechase|2000m steeplechase|2000m S\/C', na=False))
athletes.loc[mask, 'MAPPED_EVENT'] = '2000m Steeplechase'


#mask = (athletes['EVENT'].str.contains(r'Steeplechase', na=False) & athletes['DISTANCE'].str.contains(r'2000', na=False)  & athletes['DIVISION'].str.contains(r'OPEN|Open', na=False))
#athletes.loc[mask, 'MAPPED_EVENT'] = '2000m Steeplechase'


# Marathon

mask = athletes['EVENT'].str.contains(r'^Marathon$', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Marathon'
mask = athletes['EVENT'].str.contains(r'^Half\sMarathon$', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Half Marathon'
mask = athletes['EVENT'].str.contains(r'^Half\smarathon$', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Half Marathon'


# Walk

#mask = athletes['EVENT'].str.contains(r'1500m Race Walk', na=True)
#athletes.loc[mask, 'MAPPED_EVENT'] = '1500m Race Walk'
#mask = athletes['EVENT'].str.contains(r'1500 Meter Race Walk', na=True)
#athletes.loc[mask, 'MAPPED_EVENT'] = '1500m Race Walk'
#mask = (athletes['EVENT'].str.contains(r'Race Walk', na=False) & athletes['DISTANCE'].str.contains(r'1500', na=False))
#athletes.loc[mask, 'MAPPED_EVENT'] = '1500m Race Walk'


#mask = athletes['EVENT'].str.contains(r'3000m Race Walk', na=True)
#athletes.loc[mask, 'MAPPED_EVENT'] = '3000m Race Walk'
#mask = athletes['EVENT'].str.contains(r'3000 Meter Race Walk', na=True)
#athletes.loc[mask, 'MAPPED_EVENT'] = '3000m Race Walk'
#mask = (athletes['EVENT'].str.contains(r'Race Walk', na=False) & athletes['DISTANCE'].str.contains(r'3000', na=False))
#athletes.loc[mask, 'MAPPED_EVENT'] = '3000m Race Walk'


#mask = athletes['EVENT'].str.contains(r'5000 Meter Race Walk', na=True)
#athletes.loc[mask, 'MAPPED_EVENT'] = '5000m Race Walk'
#mask = athletes['EVENT'].str.contains(r'5000m Race Walk', na=True)
#athletes.loc[mask, 'MAPPED_EVENT'] = '5000m Race Walk'
mask = (athletes['EVENT'].str.contains(r'Race Walk', na=False) & athletes['DISTANCE'].str.contains(r'5000', na=False))
athletes.loc[mask, 'MAPPED_EVENT'] = '5000m Racewalk'

mask = (athletes['EVENT'].str.contains(r'Race Walk', na=False) & athletes['DISTANCE'].str.contains(r'10000', na=False))
athletes.loc[mask, 'MAPPED_EVENT'] = '10000m Racewalk'


#mask = athletes['EVENT'].str.contains(r'10000 Meter Race Walk', na=True)
#athletes.loc[mask, 'MAPPED_EVENT'] = '10000m Race Walk'

# Relay

mask = athletes['EVENT'].str.contains(r'4x80m Relay', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '4 x 80m'
mask = athletes['EVENT'].str.contains(r'^4\sx\s100m$', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '4 x 100m'
mask = athletes['EVENT'].str.contains(r'4x100m Relay', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '4 x 100m'
mask = athletes['EVENT'].str.contains(r'4 X 100m Relay', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '4 x 100m'
mask = (athletes['EVENT'].str.contains(r'Relay', na=False) & athletes['DISTANCE'].str.contains(r'400', na=False))
athletes.loc[mask, 'MAPPED_EVENT'] = '4 x 100m'

mask = athletes['EVENT'].str.contains(r'4x400m Relay', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '4 x 400m'
mask = athletes['EVENT'].str.contains(r'4 X 400m Relay', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '4 x 400m'
mask = athletes['EVENT'].str.contains(r'4x100 Meter Relay', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '4 x 100m'
mask = (athletes['EVENT'].str.contains(r'Relay', na=False) & athletes['DISTANCE'].str.contains(r'1600', na=False))
athletes.loc[mask, 'MAPPED_EVENT'] = '4 x 400m'
mask = athletes['EVENT'].str.contains(r'^4\sx\s400m$', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = '4 x 400m'

# Decathlon/Heptathlon

mask = athletes['EVENT'].str.contains(r'^Heptathlon$', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Heptathlon'
mask = athletes['EVENT'].str.contains(r'^Decathlon$', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Decathlon'
mask = athletes['EVENT'].str.contains(r'Heptathlon', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Heptathlon'
mask = athletes['EVENT'].str.contains(r'Decathlon', na=True)
athletes.loc[mask, 'MAPPED_EVENT'] = 'Decathlon'

# Map Competition Names Containing U18 into U18 Division

mask = athletes['COMPETITION'].str.contains(r'U18', na=True)
athletes.loc[mask, 'DIVISION'] = 'U18'


/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_16781/750086588.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  athletes['MAPPED_EVENT']=''
/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_16781/750086588.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  athletes[col] = athletes[col].astype(str)
/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_16781/750086588.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[

/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_8419/3513542181.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  athletes[col] = athletes[col].str.replace('\xa0', ' ', regex=True)
/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_8419/3513542181.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  athletes[col] = athletes[col].str.replace('[\x00-\x1f\x7f-\x9f]', '', regex=True)
/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_8419/3513542181.py:10: SettingWithCopyWarning: 
A 

/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_8419/3513542181.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  athletes[col] = athletes[col].str.replace('\xa0', ' ', regex=True)
/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_8419/3513542181.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  athletes[col] = athletes[col].str.replace('[\x00-\x1f\x7f-\x9f]', '', regex=True)
/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_8419/3513542181.py:10: SettingWithCopyWarning: 
A 

/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_8419/3513542181.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  athletes[col] = athletes[col].str.replace('[\x00-\x1f\x7f-\x9f]', '', regex=True)
/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_8419/3513542181.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  athletes[col] = athletes[col].str.replace('\r', ' ', regex=True)
/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_8419/3513542181.py:11: SettingWithCopyWarning: 
A v

In [589]:
for col in athletes.columns:
    athletes[col] = athletes[col].astype(str)
    athletes[col] = athletes[col].str.replace('\xa0', ' ', regex=True)
    athletes[col] = athletes[col].str.replace('[\x00-\x1f\x7f-\x9f]', '', regex=True)
    athletes[col] = athletes[col].str.replace('\r', ' ', regex=True)
    athletes[col] = athletes[col].str.replace('\n', ' ', regex=True)
    athletes[col] = athletes[col].str.strip()

/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_16781/2203170124.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  athletes[col] = athletes[col].astype(str)
/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_16781/2203170124.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  athletes[col] = athletes[col].str.replace('\xa0', ' ', regex=True)
/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_16781/2203170124.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of

/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_8419/2203170124.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  athletes[col] = athletes[col].str.replace('[\x00-\x1f\x7f-\x9f]', '', regex=True)
/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_8419/2203170124.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  athletes[col] = athletes[col].str.replace('\r', ' ', regex=True)
/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_8419/2203170124.py:6: SettingWithCopyWarning: 
A val

/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_8419/2203170124.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  athletes[col] = athletes[col].astype(str)
/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_8419/2203170124.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  athletes[col] = athletes[col].str.replace('\xa0', ' ', regex=True)
/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_8419/2203170124.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a 

In [590]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/17th SEA Youth/')


athletes.to_csv('U18_post_map_oct29.csv', sep=',', encoding='utf-8-sig', index=False)


# Process Benchmarks

In [591]:
# Load Asian Youth Benchmarks

os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/Benchmarks/')

all_benchmark = pd.read_csv("All_Benchmarks.csv")




In [592]:
all_benchmark

,BENCHMARK,EVENT,GENDER,RESULT
0,26th Asian Athletics,100m,Male,10.33
1,26th Asian Athletics,100m,Female,11.66
2,26th Asian Athletics,200m,Male,21.01
3,26th Asian Athletics,200m,Female,23.66
4,26th Asian Athletics,400m,Male,46.29
...,...,...,...,...
215,17th SEA Youth U20,Triple Jump,Female,12.57
216,17th SEA Youth U20,Discus Throw,Male,48.76
217,17th SEA Youth U20,Discus Throw,Female,40.84
218,17th SEA Youth U20,Javelin Throw,Male,62.54


In [593]:
benchmark=all_benchmark[all_benchmark['BENCHMARK']=='17th SEA Youth U18']

In [594]:
benchmark

,BENCHMARK,EVENT,GENDER,RESULT
164,17th SEA Youth U18,100m,Male,10.74
165,17th SEA Youth U18,100m,Female,12.51
166,17th SEA Youth U18,200m,Male,21.88
167,17th SEA Youth U18,200m,Female,25.71
168,17th SEA Youth U18,400m,Male,48.92
169,17th SEA Youth U18,400m,Female,58.05
170,17th SEA Youth U18,800m,Male,01:57.7
171,17th SEA Youth U18,800m,Female,02:20.6
172,17th SEA Youth U18,1500m,Male,04:23.2
173,17th SEA Youth U18,1500m,Female,05:22.1


In [595]:
benchmark=benchmark.reset_index(drop=True)

In [596]:
# Converts any time format into seconds

def convert_time(i, string, metric):

    global output
    
    l=['discus', 'throw', 'jump', 'vault', 'shot']
        
    string=string.lower()
    
   # print('metric', metric)
    
    try:
        
        if 'w' in metric:  # skip marks with illegal wind speeds
            
        #    print('W', metric)
            
            output=''
            
        else:
            
    
            if any(s in string for s in l)==True:
            
                if 'm' in metric:
            
                    metric=metric.replace('m', '')
                    output=float(str(metric))
            
                elif 'GR' in metric:
            
                    metric=metric.replace('GR', '')
                    output=float(str(metric))
                
                
                else:
    
                    output=float(str(metric))
        
            elif string=='':   # no event description at all!
                
                output='' # return nothing
            
                
        
            else:
        
                searchstring = ":"
                searchstring2 = "."
                substring=str(metric)
                count = substring.count(searchstring)
                count2 = substring.count(searchstring2)
            
                if count==0:
                
                    output=float(substring)
            
            
                elif '10,000m' in string and count==2:  # fix erroneous timing format from XX:XX:XX to XX:XX.XX
                
                
                    idx = 5 # 6th character position
                    replacement = "."
                    metric = metric[:idx] + replacement + metric[idx+1:]                
                
                    m,s = metric.split(':')            

                    output = float(datetime.timedelta(minutes=int(m),seconds=float(s)).total_seconds())

                elif '5000m' in string and count==2:  # fix erroneous timing format from XX:XX:XX to XX:XX.XX
                
                
                    idx = 5 # 6th character position
                    replacement = "."
                    metric = metric[:idx] + replacement + metric[idx+1:]                
                
                    m,s = metric.split(':')            

                    output = float(datetime.timedelta(minutes=int(m),seconds=float(s)).total_seconds())

                    
                    
                elif '1500m' in string and count==2:  # fix erroneous timing format from XX:XX:XX to XX:XX.XX
                    
                    if len(substring)==7:  # format is X:XX:XX and not XX:XX:XX 
                        
                        idx = 4 # 5th character position
                        replacement = "."
                        metric = '0' + metric[:idx] + replacement + metric[idx+1:]                
                
                        m,s = metric.split(':')            

                        output = float(datetime.timedelta(minutes=int(m),seconds=float(s)).total_seconds())
                    
                        
                    else:  # format is XX:XX:XX
                        
                        idx = 5 # 5th character position
                        replacement = "."
                        metric = metric[:idx] + replacement + metric[idx+1:]                
                
                        m,s = metric.split(':')            

                        output = float(datetime.timedelta(minutes=int(m),seconds=float(s)).total_seconds())  
             
                elif (type(metric)==datetime.time or type(metric)==datetime.datetime):
                
                                                
                    time=str(metric)
                    h, m ,s = time.split(':')
                    output = float(datetime.timedelta(hours=int(h),minutes=int(m),seconds=float(s)).total_seconds())
            
                                
                elif (count==1 and count2==1):
            
                    m,s = metric.split(':')
                    output = float(datetime.timedelta(minutes=int(m),seconds=float(s)).total_seconds())
                     
                elif (count==1 and count2==2):
                
            
                    metric = metric.replace(".", ":", 1)
            
                    h,m,s = metric.split(':')            
                    output = float(datetime.timedelta(hours=int(h),minutes=int(m),seconds=float(s)).total_seconds())
                
        
                elif (count==2 and count2==0):
                
            
                    h,m,s = metric.split(':')
                    output = float(datetime.timedelta(hours=int(h),minutes=int(m),seconds=float(s)).total_seconds())
  
            

    except:
        
        pass
                
    return output

In [597]:
def process_benchmarks(df):
    
    for i in range(len(df)):

        rowIndex = df.index[i]

        input_string=df.iloc[rowIndex,0]
    
        metric=df.iloc[rowIndex,3]
    
        if metric==None:
        
            continue
        
        out = convert_time(i, input_string, metric)
        
        print(rowIndex, input_string, out)

    
        df.loc[rowIndex, 'Metric'] = out
    
    return df

In [598]:
process_benchmarks(benchmark)

0 17th SEA Youth U18 10.74
1 17th SEA Youth U18 12.51
2 17th SEA Youth U18 21.88
3 17th SEA Youth U18 25.71
4 17th SEA Youth U18 48.92
5 17th SEA Youth U18 58.05
6 17th SEA Youth U18 117.7
7 17th SEA Youth U18 140.6
8 17th SEA Youth U18 263.2
9 17th SEA Youth U18 322.1
10 17th SEA Youth U18 557.0
11 17th SEA Youth U18 680.8
12 17th SEA Youth U18 388.9
13 17th SEA Youth U18 497.6
14 17th SEA Youth U18 13.9
15 17th SEA Youth U18 15.14
16 17th SEA Youth U18 54.81
17 17th SEA Youth U18 66.5
18 17th SEA Youth U18 1.93
19 17th SEA Youth U18 1.55
20 17th SEA Youth U18 6.76
21 17th SEA Youth U18 5.46
22 17th SEA Youth U18 14.29
23 17th SEA Youth U18 11.68
24 17th SEA Youth U18 15.23
25 17th SEA Youth U18 12.54
26 17th SEA Youth U18 47.49
27 17th SEA Youth U18 36.6
28 17th SEA Youth U18 58.87
29 17th SEA Youth U18 40.37
30 17th SEA Youth U18 53.22
31 17th SEA Youth U18 43.36
32 17th SEA Youth U18 4.08
33 17th SEA Youth U18 2.93


,BENCHMARK,EVENT,GENDER,RESULT,Metric
0,17th SEA Youth U18,100m,Male,10.74,10.74
1,17th SEA Youth U18,100m,Female,12.51,12.51
2,17th SEA Youth U18,200m,Male,21.88,21.88
3,17th SEA Youth U18,200m,Female,25.71,25.71
4,17th SEA Youth U18,400m,Male,48.92,48.92
5,17th SEA Youth U18,400m,Female,58.05,58.05
6,17th SEA Youth U18,800m,Male,01:57.7,117.70
7,17th SEA Youth U18,800m,Female,02:20.6,140.60
8,17th SEA Youth U18,1500m,Male,04:23.2,263.20
9,17th SEA Youth U18,1500m,Female,05:22.1,322.10


In [599]:
'''
for i in range(len(benchmarks)):
        
    rowIndex = benchmarks.index[i]

    input_string=benchmarks.iloc[rowIndex,0]
    
    metric=benchmarks.iloc[rowIndex,3]
    
    if metric==None:
        continue
        
    out = convert_time(i, input_string, metric)
    
    print(rowIndex, input_string, out)
     
    benchmarks.loc[rowIndex, 'Metric'] = out
'''

"\nfor i in range(len(benchmarks)):\n        \n    rowIndex = benchmarks.index[i]\n\n    input_string=benchmarks.iloc[rowIndex,0]\n    \n    metric=benchmarks.iloc[rowIndex,3]\n    \n    if metric==None:\n        continue\n        \n    out = convert_time(i, input_string, metric)\n    \n    print(rowIndex, input_string, out)\n     \n    benchmarks.loc[rowIndex, 'Metric'] = out\n"

In [600]:
benchmark

,BENCHMARK,EVENT,GENDER,RESULT,Metric
0,17th SEA Youth U18,100m,Male,10.74,10.74
1,17th SEA Youth U18,100m,Female,12.51,12.51
2,17th SEA Youth U18,200m,Male,21.88,21.88
3,17th SEA Youth U18,200m,Female,25.71,25.71
4,17th SEA Youth U18,400m,Male,48.92,48.92
5,17th SEA Youth U18,400m,Female,58.05,58.05
6,17th SEA Youth U18,800m,Male,01:57.7,117.70
7,17th SEA Youth U18,800m,Female,02:20.6,140.60
8,17th SEA Youth U18,1500m,Male,04:23.2,263.20
9,17th SEA Youth U18,1500m,Female,05:22.1,322.10


In [601]:
mask = benchmark['EVENT'].str.lower().str.contains(r'jump|throw|put|pole|decathlon|heptathlon', na=True)

benchmark.loc[mask, '2%']=benchmark['Metric']*0.98
benchmark.loc[mask, '3.5%']=benchmark['Metric']*0.965
benchmark.loc[mask, '5%']=benchmark['Metric']*0.95

benchmark.loc[~mask, '2%']=benchmark['Metric']*1.02
benchmark.loc[~mask, '3.5%']=benchmark['Metric']*1.035
benchmark.loc[~mask, '5%']=benchmark['Metric']*1.05


#benchmarks.iloc[5, [1]]='10000m run'
#benchmarks.iloc[28, [1]]='10000m run'
#benchmarks.iloc[26, [1]]='1500m'


In [602]:
benchmark['MAPPED_EVENT']=benchmark['EVENT'].str.strip()

In [603]:
for col in benchmark.columns:
    benchmark[col] = benchmark[col].astype(str)
    benchmark[col] = benchmark[col].str.replace('\xa0', ' ', regex=True)
    benchmark[col] = benchmark[col].str.replace('[\x00-\x1f\x7f-\x9f]', '', regex=True)
    benchmark[col] = benchmark[col].str.replace('\r', ' ', regex=True)
    benchmark[col] = benchmark[col].str.replace('\n', ' ', regex=True)
    benchmark[col] = benchmark[col].str.strip()


In [604]:
benchmark

,BENCHMARK,EVENT,GENDER,RESULT,Metric,2%,3.5%,5%,MAPPED_EVENT
0,17th SEA Youth U18,100m,Male,10.74,10.74,10.9548,11.1159,11.277000000000001,100m
1,17th SEA Youth U18,100m,Female,12.51,12.51,12.7602,12.947849999999999,13.1355,100m
2,17th SEA Youth U18,200m,Male,21.88,21.88,22.3176,22.645799999999998,22.974,200m
3,17th SEA Youth U18,200m,Female,25.71,25.71,26.2242,26.609849999999998,26.995500000000003,200m
4,17th SEA Youth U18,400m,Male,48.92,48.92,49.8984,50.6322,51.36600000000001,400m
5,17th SEA Youth U18,400m,Female,58.05,58.05,59.211,60.08174999999999,60.9525,400m
6,17th SEA Youth U18,800m,Male,01:57.7,117.7,120.054,121.81949999999999,123.58500000000001,800m
7,17th SEA Youth U18,800m,Female,02:20.6,140.6,143.412,145.521,147.63,800m
8,17th SEA Youth U18,1500m,Male,04:23.2,263.2,268.464,272.412,276.36,1500m
9,17th SEA Youth U18,1500m,Female,05:22.1,322.1,328.54200000000003,333.3735,338.20500000000004,1500m


In [605]:
athletes

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,DATE,YEAR,REGION,TIMESTAMP,NOW,delta_time,delta_time_conv,event_month,DOB_dt,MAPPED_EVENT
0,NG Caitlin Shan Wen,12.42,<NA>,18.5,5,<NA>,100m,<NA>,nan,<NA>,...,2025-10-23 00:00:00,2025,International,2025-10-27 21:31:00+00:00,2025-10-29 10:18:18.246231+00:00,6 days 10:18:18.246231,6,10,2010-11-24 00:00:00+00:00,100m
1,TAN Ying Tong Shannon,12.53,<NA>,18.5,2,<NA>,100m,<NA>,nan,<NA>,...,2025-10-23 00:00:00,2025,International,2025-10-27 21:31:00+00:00,2025-10-29 10:18:18.246231+00:00,6 days 10:18:18.246231,6,10,2009-11-20 00:00:00+00:00,100m
2,NG Caitlin Shan Wen,12.36,<NA>,18.5,6,<NA>,100m,<NA>,nan,<NA>,...,2025-10-23 00:00:00,2025,International,2025-10-27 21:31:00+00:00,2025-10-29 10:18:18.246231+00:00,6 days 10:18:18.246231,6,10,2010-11-24 00:00:00+00:00,100m
3,TAN Ying Tong Shannon,12.33,<NA>,18.5,7,<NA>,100m,<NA>,nan,<NA>,...,2025-10-23 00:00:00,2025,International,2025-10-27 21:31:00+00:00,2025-10-29 10:18:18.246231+00:00,6 days 10:18:18.246231,6,10,2009-11-20 00:00:00+00:00,100m
4,TAY Sean Russell,49.89,<NA>,18.5,6,<NA>,400m,<NA>,nan,<NA>,...,2025-10-24 00:00:00,2025,International,2025-10-27 21:31:00+00:00,2025-10-29 10:18:18.246231+00:00,5 days 10:18:18.246231,5,10,2009-12-25 00:00:00+00:00,400m
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
167900,Marc Brian Louis,20.89,,,1,,200m,,,,...,2025-08-11 00:00:00,2025,International,2025-08-14 12:50:00+00:00,2025-10-29 10:18:18.246231+00:00,79 days 10:18:18.246231,79,8,2002-08-07 00:00:00,200m
167922,Harry Irfan Curra,34.57,,,3,,300m,,,,...,2025-08-09 00:00:00,2025,International,2025-08-14 12:50:00+00:00,2025-10-29 10:18:18.246231+00:00,81 days 10:18:18.246231,81,8,2008-01-31 00:00:00,
168210,Rui Yong Soh,14:46.81,,,4,,5000m,,,,...,2025-08-09 00:00:00,2025,International,2025-08-14 12:50:00+00:00,2025-10-29 10:18:18.246231+00:00,81 days 10:18:18.246231,81,8,1991-08-06 00:00:00,5000m
168785,Vanessa Lee,17:11.36,<NA>,18.5,nan,<NA>,5000m,<NA>,<NA>,<NA>,...,2025-08-23 00:00:00,2025,International,2025-08-31 18:46:00+00:00,2025-10-29 10:18:18.246231+00:00,67 days 10:18:18.246231,67,8,,5000m


In [606]:
athletes[athletes['NAME']=='KO Daryen Xin Tze']

,NAME,RESULT,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,DATE,YEAR,REGION,TIMESTAMP,NOW,delta_time,delta_time_conv,event_month,DOB_dt,MAPPED_EVENT
6,KO Daryen Xin Tze,53.52,<NA>,18.5,1,<NA>,400m Hurdles,<NA>,nan,<NA>,...,2025-10-24 00:00:00,2025,International,2025-10-27 21:31:00+00:00,2025-10-29 10:18:18.246231+00:00,5 days 10:18:18.246231,5,10,2009-12-01 00:00:00+00:00,400m Hurdles


In [607]:
# There is a problem with RESULTS column being changed after this statement

#df = athletes.reset_index().merge(benchmarks.reset_index(), on=['MAPPED_EVENT','GENDER'], how='left')
#df = athletes.merge(benchmarks, on=['EVENT','GENDER'], how='left')


# Join Benchmarks to Results

In [608]:
# Merge benchmarks onto athletes on MAPPED_EVENT and GENDER

df = pd.merge(
    left=athletes, 
    right=benchmark,
    how='left',
    left_on=['MAPPED_EVENT', 'GENDER'],
    right_on=['MAPPED_EVENT', 'GENDER'],
)

In [609]:
df

,NAME,RESULT_x,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT_x,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,event_month,DOB_dt,MAPPED_EVENT,BENCHMARK,EVENT_y,RESULT_y,Metric,2%,3.5%,5%
0,NG Caitlin Shan Wen,12.42,<NA>,18.5,5,<NA>,100m,<NA>,nan,<NA>,...,10,2010-11-24 00:00:00+00:00,100m,17th SEA Youth U18,100m,12.51,12.51,12.7602,12.947849999999999,13.1355
1,TAN Ying Tong Shannon,12.53,<NA>,18.5,2,<NA>,100m,<NA>,nan,<NA>,...,10,2009-11-20 00:00:00+00:00,100m,17th SEA Youth U18,100m,12.51,12.51,12.7602,12.947849999999999,13.1355
2,NG Caitlin Shan Wen,12.36,<NA>,18.5,6,<NA>,100m,<NA>,nan,<NA>,...,10,2010-11-24 00:00:00+00:00,100m,17th SEA Youth U18,100m,12.51,12.51,12.7602,12.947849999999999,13.1355
3,TAN Ying Tong Shannon,12.33,<NA>,18.5,7,<NA>,100m,<NA>,nan,<NA>,...,10,2009-11-20 00:00:00+00:00,100m,17th SEA Youth U18,100m,12.51,12.51,12.7602,12.947849999999999,13.1355
4,TAY Sean Russell,49.89,<NA>,18.5,6,<NA>,400m,<NA>,nan,<NA>,...,10,2009-12-25 00:00:00+00:00,400m,17th SEA Youth U18,400m,48.92,48.92,49.8984,50.6322,51.36600000000001
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19391,Marc Brian Louis,20.89,,,1,,200m,,,,...,8,2002-08-07 00:00:00,200m,17th SEA Youth U18,200m,21.88,21.88,22.3176,22.645799999999998,22.974
19392,Harry Irfan Curra,34.57,,,3,,300m,,,,...,8,2008-01-31 00:00:00,,NaN,NaN,NaN,NaN,NaN,NaN,NaN
19393,Rui Yong Soh,14:46.81,,,4,,5000m,,,,...,8,1991-08-06 00:00:00,5000m,NaN,NaN,NaN,NaN,NaN,NaN,NaN
19394,Vanessa Lee,17:11.36,<NA>,18.5,nan,<NA>,5000m,<NA>,<NA>,<NA>,...,8,,5000m,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [610]:
# replace '-' with NaN

df['RESULT_x'] = df['RESULT_x'].replace(regex=r'–', value=np.NaN)
#df['SEED'] = df['SEED'].replace(regex=r'–', value=np.NaN)


In [611]:
df

,NAME,RESULT_x,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT_x,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,event_month,DOB_dt,MAPPED_EVENT,BENCHMARK,EVENT_y,RESULT_y,Metric,2%,3.5%,5%
0,NG Caitlin Shan Wen,12.42,<NA>,18.5,5,<NA>,100m,<NA>,nan,<NA>,...,10,2010-11-24 00:00:00+00:00,100m,17th SEA Youth U18,100m,12.51,12.51,12.7602,12.947849999999999,13.1355
1,TAN Ying Tong Shannon,12.53,<NA>,18.5,2,<NA>,100m,<NA>,nan,<NA>,...,10,2009-11-20 00:00:00+00:00,100m,17th SEA Youth U18,100m,12.51,12.51,12.7602,12.947849999999999,13.1355
2,NG Caitlin Shan Wen,12.36,<NA>,18.5,6,<NA>,100m,<NA>,nan,<NA>,...,10,2010-11-24 00:00:00+00:00,100m,17th SEA Youth U18,100m,12.51,12.51,12.7602,12.947849999999999,13.1355
3,TAN Ying Tong Shannon,12.33,<NA>,18.5,7,<NA>,100m,<NA>,nan,<NA>,...,10,2009-11-20 00:00:00+00:00,100m,17th SEA Youth U18,100m,12.51,12.51,12.7602,12.947849999999999,13.1355
4,TAY Sean Russell,49.89,<NA>,18.5,6,<NA>,400m,<NA>,nan,<NA>,...,10,2009-12-25 00:00:00+00:00,400m,17th SEA Youth U18,400m,48.92,48.92,49.8984,50.6322,51.36600000000001
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19391,Marc Brian Louis,20.89,,,1,,200m,,,,...,8,2002-08-07 00:00:00,200m,17th SEA Youth U18,200m,21.88,21.88,22.3176,22.645799999999998,22.974
19392,Harry Irfan Curra,34.57,,,3,,300m,,,,...,8,2008-01-31 00:00:00,,NaN,NaN,NaN,NaN,NaN,NaN,NaN
19393,Rui Yong Soh,14:46.81,,,4,,5000m,,,,...,8,1991-08-06 00:00:00,5000m,NaN,NaN,NaN,NaN,NaN,NaN,NaN
19394,Vanessa Lee,17:11.36,<NA>,18.5,nan,<NA>,5000m,<NA>,<NA>,<NA>,...,8,,5000m,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [612]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/17th SEA Youth/')


df.to_csv('U18_postmap_oct29.csv', sep=',', encoding='utf-8-sig', index=False)


In [613]:
# Convert results and seed into seconds format

df.reset_index(drop=True, inplace=True)

for col in df.columns:
    
    df[col] = df[col].astype(str)
    df[col] = df[col].str.replace('\xa0', ' ', regex=True)
    df[col] = df[col].str.replace('[\x00-\x1f\x7f-\x9f]', '', regex=True)
    df[col] = df[col].str.replace('\r', ' ', regex=True)
    df[col] = df[col].str.replace('\n', ' ', regex=True)
    df[col] = df[col].str.strip()

for i in range(len(df)):
    
    result_out=''
    
        
    rowIndex = df.index[i]

    event=df.loc[rowIndex,'MAPPED_EVENT']    # event description
    
    result=df.loc[rowIndex,'RESULT_x'] # result
    
    if result=='—' or result=='None' or result=='DQ' or result=='SCR' or result=='FS' or result=='DNQ' or result=='DNS' or result=='NH' or result=='NM' or result=='FOUL' or result=='DNF' or result=='SR' :
        continue
    
    result_out = convert_time(i, event, result)
         
    df.loc[rowIndex, 'RESULT_CONV'] = result_out



/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_16781/1944330574.py:30: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[rowIndex, 'RESULT_CONV'] = result_out


In [614]:
#df["AGE"].fillna(0, inplace=True)
#df['AGE'] = df['AGE'].astype('float')

In [615]:
df

,NAME,RESULT_x,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT_x,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,DOB_dt,MAPPED_EVENT,BENCHMARK,EVENT_y,RESULT_y,Metric,2%,3.5%,5%,RESULT_CONV
0,NG Caitlin Shan Wen,12.42,<NA>,18.5,5,<NA>,100m,<NA>,nan,<NA>,...,2010-11-24 00:00:00+00:00,100m,17th SEA Youth U18,100m,12.51,12.51,12.7602,12.947849999999999,13.1355,12.42
1,TAN Ying Tong Shannon,12.53,<NA>,18.5,2,<NA>,100m,<NA>,nan,<NA>,...,2009-11-20 00:00:00+00:00,100m,17th SEA Youth U18,100m,12.51,12.51,12.7602,12.947849999999999,13.1355,12.53
2,NG Caitlin Shan Wen,12.36,<NA>,18.5,6,<NA>,100m,<NA>,nan,<NA>,...,2010-11-24 00:00:00+00:00,100m,17th SEA Youth U18,100m,12.51,12.51,12.7602,12.947849999999999,13.1355,12.36
3,TAN Ying Tong Shannon,12.33,<NA>,18.5,7,<NA>,100m,<NA>,nan,<NA>,...,2009-11-20 00:00:00+00:00,100m,17th SEA Youth U18,100m,12.51,12.51,12.7602,12.947849999999999,13.1355,12.33
4,TAY Sean Russell,49.89,<NA>,18.5,6,<NA>,400m,<NA>,nan,<NA>,...,2009-12-25 00:00:00+00:00,400m,17th SEA Youth U18,400m,48.92,48.92,49.8984,50.6322,51.36600000000001,49.89
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19391,Marc Brian Louis,20.89,,,1,,200m,,,,...,2002-08-07 00:00:00,200m,17th SEA Youth U18,200m,21.88,21.88,22.3176,22.645799999999998,22.974,20.89
19392,Harry Irfan Curra,34.57,,,3,,300m,,,,...,2008-01-31 00:00:00,,nan,nan,nan,nan,nan,nan,nan,
19393,Rui Yong Soh,14:46.81,,,4,,5000m,,,,...,1991-08-06 00:00:00,5000m,nan,nan,nan,nan,nan,nan,nan,886.81
19394,Vanessa Lee,17:11.36,<NA>,18.5,nan,<NA>,5000m,<NA>,<NA>,<NA>,...,,5000m,nan,nan,nan,nan,nan,nan,nan,1031.36


In [616]:
# Choose SEED if better than RESULT

#condition1=df['SEED_CONV']>df['RESULT_CONV']
#condition2=((df['CATEGORY_EVENT']=='Jump')|(df['CATEGORY_EVENT']=='Throw'))
#condition3=df['SEED_CONV']<df['RESULT_CONV']
#condition4=~((df['CATEGORY_EVENT']=='Jump')|(df['CATEGORY_EVENT']=='Throw'))


#df['RESULT_BEST']=df['SEED_CONV'].where((condition1 & condition2)|(condition3 & condition4), df['RESULT_CONV'].values)

df['RESULT_BEST'] = df['RESULT_CONV']

In [617]:
df

,NAME,RESULT_x,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT_x,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,MAPPED_EVENT,BENCHMARK,EVENT_y,RESULT_y,Metric,2%,3.5%,5%,RESULT_CONV,RESULT_BEST
0,NG Caitlin Shan Wen,12.42,<NA>,18.5,5,<NA>,100m,<NA>,nan,<NA>,...,100m,17th SEA Youth U18,100m,12.51,12.51,12.7602,12.947849999999999,13.1355,12.42,12.42
1,TAN Ying Tong Shannon,12.53,<NA>,18.5,2,<NA>,100m,<NA>,nan,<NA>,...,100m,17th SEA Youth U18,100m,12.51,12.51,12.7602,12.947849999999999,13.1355,12.53,12.53
2,NG Caitlin Shan Wen,12.36,<NA>,18.5,6,<NA>,100m,<NA>,nan,<NA>,...,100m,17th SEA Youth U18,100m,12.51,12.51,12.7602,12.947849999999999,13.1355,12.36,12.36
3,TAN Ying Tong Shannon,12.33,<NA>,18.5,7,<NA>,100m,<NA>,nan,<NA>,...,100m,17th SEA Youth U18,100m,12.51,12.51,12.7602,12.947849999999999,13.1355,12.33,12.33
4,TAY Sean Russell,49.89,<NA>,18.5,6,<NA>,400m,<NA>,nan,<NA>,...,400m,17th SEA Youth U18,400m,48.92,48.92,49.8984,50.6322,51.36600000000001,49.89,49.89
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19391,Marc Brian Louis,20.89,,,1,,200m,,,,...,200m,17th SEA Youth U18,200m,21.88,21.88,22.3176,22.645799999999998,22.974,20.89,20.89
19392,Harry Irfan Curra,34.57,,,3,,300m,,,,...,,nan,nan,nan,nan,nan,nan,nan,,
19393,Rui Yong Soh,14:46.81,,,4,,5000m,,,,...,5000m,nan,nan,nan,nan,nan,nan,nan,886.81,886.81
19394,Vanessa Lee,17:11.36,<NA>,18.5,nan,<NA>,5000m,<NA>,<NA>,<NA>,...,5000m,nan,nan,nan,nan,nan,nan,nan,1031.36,1031.36


In [618]:
# Change to numeric

df[['2%', '3.5%', '5%', 'RESULT_BEST', 'Metric']] = df[['2%', '3.5%', '5%', 'RESULT_BEST', 'Metric']].apply(pd.to_numeric, errors='coerce')

In [619]:
mask = df['CATEGORY_EVENT'].str.lower().str.contains(r'jump|throw|decathlon|heptathlon', na=True)

df.loc[mask, 'Delta2'] = df['RESULT_BEST']-df['2%']
df.loc[mask, 'Delta3.5'] = df['RESULT_BEST']-df['3.5%']
df.loc[mask, 'Delta5'] = df['RESULT_BEST']-df['5%']
df.loc[mask, 'Delta_Benchmark'] = df['RESULT_BEST']-df['Metric']

df.loc[~mask, 'Delta2'] =  df['2%'] - df['RESULT_BEST']
df.loc[~mask, 'Delta3.5'] = df['3.5%'] - df['RESULT_BEST']
df.loc[~mask, 'Delta5'] = df['5%'] - df['RESULT_BEST']
df.loc[~mask, 'Delta_Benchmark'] = df['Metric'] - df['RESULT_BEST']

#rslt_df['Delta2']=rslt_df['2pc']-rslt_df['RESULT_CONV']
#rslt_df['Delta35']=rslt_df['35pc']-rslt_df['RESULT_CONV']
#rslt_df['Delta5']=rslt_df['5pc']-rslt_df['RESULT_CONV']
df=df.loc[df['COMPETITION']!='Southeast Asian Games'] # Do not include results from SEAG in dataset

In [620]:
# Performance metric to filter out athletes

df['PERF_SCALAR']=df['Delta5']/df['Metric']*100

In [621]:
#df[df['NAME']=='MOHAMED IMRAN Ahmad Zubayr Bin']

In [622]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/17th SEA Youth/')


df.to_csv('U18_postmap_benchmarked_oct29.csv', sep=',', encoding='utf-8-sig', index=False)


In [623]:
df[df['MAPPED_EVENT']=='10,000m']

,NAME,RESULT_x,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT_x,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,2%,3.5%,5%,RESULT_CONV,RESULT_BEST,Delta2,Delta3.5,Delta5,Delta_Benchmark,PERF_SCALAR
296,"Ko, Keane",34:03.99,Club Zoom,12,1,Open,Run,10000.0,None,K326A01,...,NaN,NaN,NaN,2043.99,2043.99,NaN,NaN,NaN,NaN,NaN
297,"Tea, Wee Keat",39:35.86,Lacticbuds,12,5,Open,Run,10000.0,None,W549C02,...,NaN,NaN,NaN,2375.86,2375.86,NaN,NaN,NaN,NaN,NaN
298,"Ho, Sheng Hui",38:37.79,Singapore University of Social,12,4,Open,Run,10000.0,None,S039Z01,...,NaN,NaN,NaN,2317.79,2317.79,NaN,NaN,NaN,NaN,NaN
299,"SAVIOUR, GILBERT DOMINIC",37:32.24,Cougars Athletic Association,12,3,Open,Run,10000.0,None,G797C05,...,NaN,NaN,NaN,2252.24,2252.24,NaN,NaN,NaN,NaN,NaN
300,"Chin, Matthew",39:55.64,Nanyang Technological Uni,12,6,Open,Run,10000.0,None,M886G01,...,NaN,NaN,NaN,2395.64,2395.64,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15624,"Tan, Bernice",44:25.6,Lacticbuds,12,3,Open,Run,10000,None,B075C00,...,NaN,NaN,NaN,2665.6,2665.60,NaN,NaN,NaN,NaN,NaN
15628,"Goh, Shing Ling",39:21.6,TeamFabian,12,2,Open,Run,10000,None,S268G99,...,NaN,NaN,NaN,2361.6,2361.60,NaN,NaN,NaN,NaN,NaN
15646,"Sng, Raymond",35:28.2,Club Zoom,12,2,Open,Run,10000,None,R559H98,...,NaN,NaN,NaN,2128.2,2128.20,NaN,NaN,NaN,NaN,NaN
19381,Shaun Goh,31:02.40,,,5,,"10,000m",,,,...,NaN,NaN,NaN,1862.4,1862.40,NaN,NaN,NaN,NaN,NaN


In [624]:
# Strip out punctuation from NAME
from itertools import permutations
import string
from dateutil import parser

translator = str.maketrans('', '', string.punctuation)
df['clean_name'] = df['NAME'].str.translate(translator)
df['clean_name'] = df['clean_name'].str.casefold()


df = df.reset_index(drop=True)

translator = str.maketrans('', '', string.punctuation)

df['clean_name'] = df['NAME'].apply(lambda x: str(x).translate(translator))
df['clean_name'] = df['clean_name'].str.casefold()


In [625]:
def parse_dob(x):
    if pd.isna(x) or str(x).strip().lower() in ["none", "nan", "nat", ""]:
        return None
    try:
        # force dayfirst since your DOBs are in DD/MM/YY
        dt = parser.parse(str(x), dayfirst=True)
        # fix 2-digit year misinterpretation (like 1912 instead of 2012)
        if dt.year < 1950:  
            dt = dt.replace(year=dt.year + 100)
        return dt.strftime("%Y-%m-%d")
    except Exception:
        return None

df["DOB_parsed"] = df["DOB"].apply(parse_dob)

# Build dictionary without null DOBs
dictionary_dob_clean_name = {
    row["clean_name"]: row["DOB_parsed"]
    for _, row in df.iterrows()
    if row["DOB_parsed"] is not None
}

# Test Aarika Ray
print(dictionary_dob_clean_name.get("aarika ray"))

2012-05-19


In [626]:
'''
# Read a variation name list and corrections from CSVs

os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/Name Variations/')

names = pd.read_csv("junior_names.csv")

for index, row in names.iterrows():
        
    print(names.VARIATION, names.NAME)
    df['NAME_MAPPED'] = df['NAME'].replace(regex=rf"{row['VARIATION']}", value=f"{row['NAME']}")
'''

'\n# Read a variation name list and corrections from CSVs\n\nos.chdir(\'/Users/veesheenyuen/Desktop/DataScience/SAA/Name Variations/\')\n\nnames = pd.read_csv("junior_names.csv")\n\nfor index, row in names.iterrows():\n        \n    print(names.VARIATION, names.NAME)\n    df[\'NAME_MAPPED\'] = df[\'NAME\'].replace(regex=rf"{row[\'VARIATION\']}", value=f"{row[\'NAME\']}")\n'

In [627]:
# Read name variations from GCS name lists bucket (OLD VERSION)

'''
df['NAME'] = df['NAME'].str.replace('\xa0', '', regex=True)
df['NAME'] = df['NAME'].str.replace('[\x00-\x1f\x7f-\x9f]', '', regex=True)
df['NAME'] = df['NAME'].str.replace('\r', '', regex=True)
df['NAME'] = df['NAME'].str.replace('\n', '', regex=True)
df['NAME'] = df['NAME'].str.strip()

df['NAME'] = df['NAME'].str.casefold()  # everything lower case


# Read csv from GCS bucket

file_path = "gs://name_variations/name_variations.csv"
names = pd.read_csv(file_path,
                 sep=",",
                 storage_options={"token": '/Users/veesheenyuen/Desktop/DataScience/Keys/saa-analytics-7c8937b70609.json'})


# Clean and apply casefold to name variations

names['NAME'] = names['NAME'].str.replace('\xa0', '', regex=True)
names['NAME'] = names['NAME'].str.replace('[\x00-\x1f\x7f-\x9f]', '', regex=True)
names['NAME'] = names['NAME'].str.replace('\r', '', regex=True)
names['NAME'] = names['NAME'].str.replace('\n', '', regex=True)
names['NAME'] = names['NAME'].str.strip()
names['VARIATION'] = names['VARIATION'].str.replace('\xa0', '', regex=True)
names['VARIATION'] = names['VARIATION'].str.replace('[\x00-\x1f\x7f-\x9f]', '', regex=True)
names['VARIATION'] = names['VARIATION'].str.replace('\r', '', regex=True)
names['VARIATION'] = names['VARIATION'].str.replace('\n', '', regex=True)
names['VARIATION'] = names['VARIATION'].str.strip()

names['VARIATION'] = names['VARIATION'].str.casefold() # convert to lower case
names['NAME'] = names['NAME'].str.casefold()


# Iterate over dataframe and replace names

for index, row in names.iterrows():
        
    df['NAME'] = df['NAME'].replace(regex=rf"{row['VARIATION']}", value=f"{row['NAME']}")
    

    
df['NAME'] = df['NAME'].str.title()  # capitalize first letter
'''

'\ndf[\'NAME\'] = df[\'NAME\'].str.replace(\'\xa0\', \'\', regex=True)\ndf[\'NAME\'] = df[\'NAME\'].str.replace(\'[\x00-\x1f\x7f-\x9f]\', \'\', regex=True)\ndf[\'NAME\'] = df[\'NAME\'].str.replace(\'\r\', \'\', regex=True)\ndf[\'NAME\'] = df[\'NAME\'].str.replace(\'\n\', \'\', regex=True)\ndf[\'NAME\'] = df[\'NAME\'].str.strip()\n\ndf[\'NAME\'] = df[\'NAME\'].str.casefold()  # everything lower case\n\n\n# Read csv from GCS bucket\n\nfile_path = "gs://name_variations/name_variations.csv"\nnames = pd.read_csv(file_path,\n                 sep=",",\n                 storage_options={"token": \'/Users/veesheenyuen/Desktop/DataScience/Keys/saa-analytics-7c8937b70609.json\'})\n\n\n# Clean and apply casefold to name variations\n\nnames[\'NAME\'] = names[\'NAME\'].str.replace(\'\xa0\', \'\', regex=True)\nnames[\'NAME\'] = names[\'NAME\'].str.replace(\'[\x00-\x1f\x7f-\x9f]\', \'\', regex=True)\nnames[\'NAME\'] = names[\'NAME\'].str.replace(\'\r\', \'\', regex=True)\nnames[\'NAME\'] = names[\'NAM

In [628]:
# Fastest execution speed version

# Normalize function as before
def normalize_text(s):
    return (str(s)
            .replace('\xa0', '')
            .replace('\r', '')
            .replace('\n', '')
            .strip()
            .casefold())

# Normalize dataframe
df['NAME'] = df['NAME'].apply(normalize_text)

# Load variations file and normalize
file_path = "gs://name_variations/name_variations.csv"
names = pd.read_csv(file_path,
                    sep=',',
                    storage_options={"token": '/Users/veesheenyuen/Desktop/DataScience/Keys/saa-analytics-7c8937b70609.json'})

names['VARIATION'] = names['VARIATION'].apply(normalize_text)
names['NAME'] = names['NAME'].apply(normalize_text)

# Precompile all regex patterns safely

compiled_patterns = []
for pattern_str, replacement in zip(names['VARIATION'], names['NAME']):
    try:
        compiled_re = re.compile(pattern_str)
        compiled_patterns.append( (compiled_re, replacement) )
    except re.error as e:
        print(f"Skipping invalid regex pattern: {pattern_str} Error: {e}")

# Iterate over all patterns and apply replacements using precompiled regexes
for regex, replacement in compiled_patterns:
    df['NAME'] = df['NAME'].str.replace(regex, replacement, regex=True)

# Capitalize final standardized names
df['NAME'] = df['NAME'].str.title()


In [629]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/Asian Youth/')


df.to_csv("names_check.csv", encoding='utf-8')

In [630]:
# Exclude foreigners from MALAYSIA, THAILAND etc.

#df_select = df[(df['TEAM']!='Malaysia') & (df['TEAM']!='THAILAND') & (df['TEAM']!='China') & (df['TEAM']!='South Korea') & (df['TEAM']!='Laos') & (df['TEAM']!='Philippines') & (df['TEAM']!='Piboonbumpen Thailand') & (df['TEAM']!='Chinese Taipei') & (df['TEAM']!='Gurkha Contingent') & (df['TEAM']!='Australia') & (df['TEAM']!='Piboonbumpen Thailand') & (df['TEAM']!='Hong Kong') & (df['TEAM']!='PERAK')] 

df_select = df[(df['TEAM']!='Malaysia')&(df['TEAM']!='THAILAND')&(df['TEAM']!='China')&(df['TEAM']!='Thailand') 
                       &(df['TEAM']!='South Korea')&(df['TEAM']!='Laos')&(df['TEAM']!='Myanmar') 
                       &(df['TEAM']!='Philippines')&(df['TEAM']!='Piboonbumpen Thailand') 
                       &(df['TEAM']!='Chinese Taipei')&(df['TEAM']!='Gurkha Contingent') 
                       &(df['TEAM']!='Australia')&(df['TEAM']!='Piboonbumpen Thailand') 
                       &(df['TEAM']!='Hong Kong')&(df['TEAM']!='PERAK')&(df['TEAM']!='Sri Lanka') 
                       &(df['TEAM']!='Indonesia')&(df['TEAM']!='THAILAND')&(df['TEAM']!='MALAYSIA') 
                       &(df['TEAM']!='PHILIPPINES') & (df['TEAM']!='SOUTH KOREA')&(df['TEAM']!='Waseda') 
                       &(df['TEAM']!='LAOS')&(df['TEAM']!='CHINESE TAIPEI')&(df['TEAM']!='Vietnam')
                       &(df['TEAM']!='INDIA')&(df['TEAM']!='Hong Kong, China')&(df['TEAM']!='AIC JAPAN')
                       &(df['NATIONALITY']!='GBR')&(df['NATIONALITY']!='JPN')&(df['NATIONALITY']!='SRI')&(df['NATIONALITY']!='SAM')
                       &(df['NATIONALITY']!='THA')&(df['NATIONALITY']!='IND')&(df['NATIONALITY']!='PHI')&(df['NATIONALITY']!='MAS')] 

In [631]:
df_select

,NAME,RESULT_x,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT_x,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,5%,RESULT_CONV,RESULT_BEST,Delta2,Delta3.5,Delta5,Delta_Benchmark,PERF_SCALAR,clean_name,DOB_parsed
0,Ng Caitlin Shan Wen,12.42,<NA>,18.5,5,<NA>,100m,<NA>,nan,<NA>,...,13.1355,12.42,12.42,0.3402,0.52785,0.7155,0.09,5.719424,ng caitlin shan wen,2010-11-24
1,Tan Ying Tong Shannon,12.53,<NA>,18.5,2,<NA>,100m,<NA>,nan,<NA>,...,13.1355,12.53,12.53,0.2302,0.41785,0.6055,-0.02,4.840128,tan ying tong shannon,2009-11-20
2,Ng Caitlin Shan Wen,12.36,<NA>,18.5,6,<NA>,100m,<NA>,nan,<NA>,...,13.1355,12.36,12.36,0.4002,0.58785,0.7755,0.15,6.199041,ng caitlin shan wen,2010-11-24
3,Tan Ying Tong Shannon,12.33,<NA>,18.5,7,<NA>,100m,<NA>,nan,<NA>,...,13.1355,12.33,12.33,0.4302,0.61785,0.8055,0.18,6.438849,tan ying tong shannon,2009-11-20
4,Tay Sean Russell,49.89,<NA>,18.5,6,<NA>,400m,<NA>,nan,<NA>,...,51.3660,49.89,49.89,0.0084,0.74220,1.4760,-0.97,3.017171,tay sean russell,2009-12-25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19391,Louis Marc Brian,20.89,,,1,,200m,,,,...,22.9740,20.89,20.89,1.4276,1.75580,2.0840,0.99,9.524680,marc brian louis,2002-08-07
19392,Curran Harry Irfan,34.57,,,3,,300m,,,,...,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,harry irfan curra,2008-01-31
19393,Soh Rui Yong Guillaume,14:46.81,,,4,,5000m,,,,...,NaN,886.81,886.81,NaN,NaN,NaN,NaN,NaN,rui yong soh,1991-08-06
19394,Vanessa Lee,17:11.36,<NA>,18.5,nan,<NA>,5000m,<NA>,<NA>,<NA>,...,NaN,1031.36,1031.36,NaN,NaN,NaN,NaN,NaN,vanessa lee,None


In [632]:
'''
# Read list of foreigners

foreigners = pd.read_csv('/Users/veesheenyuen/Desktop/DataScience/SAA/MM/List of Foreigners.csv', encoding='latin-1')
'''


"\n# Read list of foreigners\n\nforeigners = pd.read_csv('/Users/veesheenyuen/Desktop/DataScience/SAA/MM/List of Foreigners.csv', encoding='latin-1')\n"

In [633]:
# Read list of foreigners from GCS bucket

file_path = "gs://name_lists/List of Foreigners.csv"
foreigners = pd.read_csv(file_path,
                 sep=",",
                 encoding="unicode escape",
                 storage_options={"token": '/Users/veesheenyuen/Desktop/DataScience/Keys/saa-analytics-7c8937b70609.json'})


In [634]:
foreigners

,LAST_NAME,FIRST_NAME
0,Aaryan,Greuter Christoph
1,Akahodani,Takayuki
2,Apondar,Audric
3,Brooks,Ruby
4,Brouwer,Cees
...,...,...
235,Kashama,Biwesa Daniel
236,ISMAIL,MUHAMMAD ZULFIQAR
237,Jayaganeson,Kirtisha
238,LIN,Yu Sian


In [635]:
foreigners['V1'] = foreigners['LAST_NAME']+' '+foreigners['FIRST_NAME']
foreigners['V2'] = foreigners['FIRST_NAME']+' '+foreigners['LAST_NAME']
foreigners['V3'] = foreigners['LAST_NAME']+', '+foreigners['FIRST_NAME']
foreigners['V4'] = foreigners['FIRST_NAME']+' '+foreigners['LAST_NAME']

for1 = foreigners['V1'].dropna().tolist()
for2 = foreigners['V2'].dropna().tolist()
for3 = foreigners['V3'].dropna().tolist()
for4 = foreigners['V4'].dropna().tolist()

foreign_list = for1+for2+for3+for4 

foreign_list_casefold=[s.casefold() for s in foreign_list]

exclusions = foreign_list_casefold

no_foreigners_list = df_select.loc[~df['NAME'].str.casefold().isin(exclusions)]  # ~ means NOT IN. DROP spex carded athletes

In [636]:
no_foreigners_list

,NAME,RESULT_x,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT_x,DISTANCE,EVENT_CLASS,UNIQUE_ID,...,5%,RESULT_CONV,RESULT_BEST,Delta2,Delta3.5,Delta5,Delta_Benchmark,PERF_SCALAR,clean_name,DOB_parsed
0,Ng Caitlin Shan Wen,12.42,<NA>,18.5,5,<NA>,100m,<NA>,nan,<NA>,...,13.1355,12.42,12.42,0.3402,0.52785,0.7155,0.09,5.719424,ng caitlin shan wen,2010-11-24
1,Tan Ying Tong Shannon,12.53,<NA>,18.5,2,<NA>,100m,<NA>,nan,<NA>,...,13.1355,12.53,12.53,0.2302,0.41785,0.6055,-0.02,4.840128,tan ying tong shannon,2009-11-20
2,Ng Caitlin Shan Wen,12.36,<NA>,18.5,6,<NA>,100m,<NA>,nan,<NA>,...,13.1355,12.36,12.36,0.4002,0.58785,0.7755,0.15,6.199041,ng caitlin shan wen,2010-11-24
3,Tan Ying Tong Shannon,12.33,<NA>,18.5,7,<NA>,100m,<NA>,nan,<NA>,...,13.1355,12.33,12.33,0.4302,0.61785,0.8055,0.18,6.438849,tan ying tong shannon,2009-11-20
4,Tay Sean Russell,49.89,<NA>,18.5,6,<NA>,400m,<NA>,nan,<NA>,...,51.3660,49.89,49.89,0.0084,0.74220,1.4760,-0.97,3.017171,tay sean russell,2009-12-25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19391,Louis Marc Brian,20.89,,,1,,200m,,,,...,22.9740,20.89,20.89,1.4276,1.75580,2.0840,0.99,9.524680,marc brian louis,2002-08-07
19392,Curran Harry Irfan,34.57,,,3,,300m,,,,...,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,harry irfan curra,2008-01-31
19393,Soh Rui Yong Guillaume,14:46.81,,,4,,5000m,,,,...,NaN,886.81,886.81,NaN,NaN,NaN,NaN,NaN,rui yong soh,1991-08-06
19394,Vanessa Lee,17:11.36,<NA>,18.5,nan,<NA>,5000m,<NA>,<NA>,<NA>,...,NaN,1031.36,1031.36,NaN,NaN,NaN,NaN,NaN,vanessa lee,None


In [637]:
# Choose the best result for each event participated by every athlete

top_performers_clean = no_foreigners_list.sort_values(['MAPPED_EVENT', 'NAME','PERF_SCALAR'],ascending=False).groupby(['MAPPED_EVENT', 'NAME']).head(1)


In [638]:
top_performers_clean.reset_index(inplace=True)


In [639]:
'''

# Process dates to extract age

# Map NSG divisions into age

mask = (top_performers_clean['DIVISION'].str.contains(r'A', na=False))
top_performers_clean.loc[mask, 'AGE'] = '18.5'

mask = (top_performers_clean['DIVISION'].str.contains(r'B', na=False))
top_performers_clean.loc[mask, 'AGE'] = '16'

mask = (top_performers_clean['DIVISION'].str.contains(r'C', na=False))
top_performers_clean.loc[mask, 'AGE'] = '13.5'

mask = (top_performers_clean['DIVISION'].str.contains(r'O', na=False))
top_performers_clean.loc[mask, 'AGE'] = '12'
'''

"\n\n# Process dates to extract age\n\n# Map NSG divisions into age\n\nmask = (top_performers_clean['DIVISION'].str.contains(r'A', na=False))\ntop_performers_clean.loc[mask, 'AGE'] = '18.5'\n\nmask = (top_performers_clean['DIVISION'].str.contains(r'B', na=False))\ntop_performers_clean.loc[mask, 'AGE'] = '16'\n\nmask = (top_performers_clean['DIVISION'].str.contains(r'C', na=False))\ntop_performers_clean.loc[mask, 'AGE'] = '13.5'\n\nmask = (top_performers_clean['DIVISION'].str.contains(r'O', na=False))\ntop_performers_clean.loc[mask, 'AGE'] = '12'\n"

In [640]:
'''

def length(string):

    B = ''
    year = ''

    try:

        length = len(string)

        if length == 2:

            string = '19' + string

        elif length == 1:

            string = ''

        else:

            pass

        if string is not None or len(string) != 1:

            B = parser.parse(string, dayfirst=True)
                        
    except:

        pass

    return B


top_performers_clean['DOB_new'] = top_performers_clean['DOB'].apply(length)

'''


"\n\ndef length(string):\n\n    B = ''\n    year = ''\n\n    try:\n\n        length = len(string)\n\n        if length == 2:\n\n            string = '19' + string\n\n        elif length == 1:\n\n            string = ''\n\n        else:\n\n            pass\n\n        if string is not None or len(string) != 1:\n\n            B = parser.parse(string, dayfirst=True)\n                        \n    except:\n\n        pass\n\n    return B\n\n\ntop_performers_clean['DOB_new'] = top_performers_clean['DOB'].apply(length)\n\n"

In [641]:
'''

top_performers_clean['DOB_new'] = pd.to_datetime(top_performers_clean['DOB_new'], errors='coerce')

top_performers_clean['year_extract'] = top_performers_clean['DOB_new'].dt.strftime('%Y')

top_performers_clean['year_extract'] = pd.to_numeric(top_performers_clean['year_extract'])

top_performers_clean['age_extract'] = 2025 - top_performers_clean['year_extract']
'''

"\n\ntop_performers_clean['DOB_new'] = pd.to_datetime(top_performers_clean['DOB_new'], errors='coerce')\n\ntop_performers_clean['year_extract'] = top_performers_clean['DOB_new'].dt.strftime('%Y')\n\ntop_performers_clean['year_extract'] = pd.to_numeric(top_performers_clean['year_extract'])\n\ntop_performers_clean['age_extract'] = 2025 - top_performers_clean['year_extract']\n"

In [642]:
'''

def age(number):  # correct negative age numbers
    
    if number<0:
        
        number+=100
        
    return number


top_performers_clean['age_extract']=top_performers_clean['age_extract'].apply(age)
'''

"\n\ndef age(number):  # correct negative age numbers\n    \n    if number<0:\n        \n        number+=100\n        \n    return number\n\n\ntop_performers_clean['age_extract']=top_performers_clean['age_extract'].apply(age)\n"

In [643]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/17th SEA Youth/')


top_performers_clean.to_csv('U18_top_performers_oct29.csv', encoding='utf-8')

In [644]:
top_performers_clean

,index,NAME,RESULT_x,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT_x,DISTANCE,EVENT_CLASS,...,5%,RESULT_CONV,RESULT_BEST,Delta2,Delta3.5,Delta5,Delta_Benchmark,PERF_SCALAR,clean_name,DOB_parsed
0,16588,Zhou Xuanyu,9.12,DHS,13.5,10.0,C,Triple Jump,None,None,...,11.0960,9.12,9.12,-2.3264,-2.15120,-1.9760,-2.56,-16.917808,zhou xuanyu,None
1,16656,Zhong Chuhan,12.26,None,None,None,None,Triple Jump,None,None,...,11.0960,12.26,12.26,0.8136,0.98880,1.1640,0.58,9.965753,chuhan zhong,2005-10-18
2,16364,Zheng Justin De,10.47m,National Junior College,14,12,U15,Triple Jump,0,None,...,13.5755,10.47,10.47,-3.5342,-3.31985,-3.1055,-3.82,-21.731980,zheng justin de,2011-02-21
3,16497,"Zhao, Daniel",11.67m,Hwa Chong Institution,14,3,U15,Triple Jump,0,None,...,13.5755,11.67,11.67,-2.3342,-2.11985,-1.9055,-2.62,-13.334500,zhao daniel,2011-07-25
4,18293,Zhao Daniel,11.98,-,,1,,Triple Jump,,nan,...,13.5755,11.98,11.98,-2.0242,-1.80985,-1.5955,-2.31,-11.165150,daniel zhao,2011-07-25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12505,7003,"., Shaik Isa",15.49,Club ZOOM,8,32,U9,Dash,80,,...,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,shaik isa,2016-11-21
12506,6660,"., Nur Elena",16.42,UNA,6,1,U7,Dash,80,,...,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,nur elena,2018-12-04
12507,13258,"., Dharkshitha",08:35.6,Cedar Girls Secondary School,15,1,U18,Race Walk,1500,None,...,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,dharkshitha,2010-06-04
12508,12881,"., Cayden",06:05.1,North Vista,12,54,Open,Mile Run,1,,...,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,cayden,2010-01-01


In [645]:
# Choose best performance for each event

#tiered_performers = top_performers_clean.sort_values(['GENDER', 'MAPPED_EVENT', 'PERF_SCALAR'],ascending=False).groupby(['MAPPED_EVENT', 'NAME']).head(1)

tiered_performers = top_performers_clean


In [646]:
tiered_performers

,index,NAME,RESULT_x,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT_x,DISTANCE,EVENT_CLASS,...,5%,RESULT_CONV,RESULT_BEST,Delta2,Delta3.5,Delta5,Delta_Benchmark,PERF_SCALAR,clean_name,DOB_parsed
0,16588,Zhou Xuanyu,9.12,DHS,13.5,10.0,C,Triple Jump,None,None,...,11.0960,9.12,9.12,-2.3264,-2.15120,-1.9760,-2.56,-16.917808,zhou xuanyu,None
1,16656,Zhong Chuhan,12.26,None,None,None,None,Triple Jump,None,None,...,11.0960,12.26,12.26,0.8136,0.98880,1.1640,0.58,9.965753,chuhan zhong,2005-10-18
2,16364,Zheng Justin De,10.47m,National Junior College,14,12,U15,Triple Jump,0,None,...,13.5755,10.47,10.47,-3.5342,-3.31985,-3.1055,-3.82,-21.731980,zheng justin de,2011-02-21
3,16497,"Zhao, Daniel",11.67m,Hwa Chong Institution,14,3,U15,Triple Jump,0,None,...,13.5755,11.67,11.67,-2.3342,-2.11985,-1.9055,-2.62,-13.334500,zhao daniel,2011-07-25
4,18293,Zhao Daniel,11.98,-,,1,,Triple Jump,,nan,...,13.5755,11.98,11.98,-2.0242,-1.80985,-1.5955,-2.31,-11.165150,daniel zhao,2011-07-25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12505,7003,"., Shaik Isa",15.49,Club ZOOM,8,32,U9,Dash,80,,...,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,shaik isa,2016-11-21
12506,6660,"., Nur Elena",16.42,UNA,6,1,U7,Dash,80,,...,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,nur elena,2018-12-04
12507,13258,"., Dharkshitha",08:35.6,Cedar Girls Secondary School,15,1,U18,Race Walk,1500,None,...,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,dharkshitha,2010-06-04
12508,12881,"., Cayden",06:05.1,North Vista,12,54,Open,Mile Run,1,,...,NaN,,NaN,NaN,NaN,NaN,NaN,NaN,cayden,2010-01-01


In [647]:
# Identify Tier 1/2/3 performers

#top_performers_clean['TIER'] = np.where((top_performers_clean['Delta_Benchmark']>=0), 'Tier 1',    
#                                np.where(((top_performers_clean['Delta_Benchmark']<0) & (top_performers_clean['Delta2']>=0)), 'Tier2',
#                                np.where(((top_performers_clean['Delta2']<0) & (top_performers_clean['Delta3.5']>=0)), 'Tier3', ' ')))


tiered_performers['TIER'] = np.where((tiered_performers['Delta_Benchmark']>=0), 'Tier 1',    
                                np.where(((tiered_performers['Delta_Benchmark']<0) & (tiered_performers['Delta2']>=0)), 'Tier 2',
                                np.where(((tiered_performers['Delta2']<0) & (tiered_performers['Delta3.5']>=0)), 'Tier 3', 
                                np.where(((tiered_performers['Delta3.5']<0) & (tiered_performers['Delta5']>=0)), 'Tier 4', ' '))))



In [648]:
# Drop rows without a benchmark

final_df = tiered_performers[tiered_performers['BENCHMARK'].notna()]


In [649]:
final_df

,index,NAME,RESULT_x,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT_x,DISTANCE,EVENT_CLASS,...,RESULT_CONV,RESULT_BEST,Delta2,Delta3.5,Delta5,Delta_Benchmark,PERF_SCALAR,clean_name,DOB_parsed,TIER
0,16588,Zhou Xuanyu,9.12,DHS,13.5,10.0,C,Triple Jump,None,None,...,9.12,9.12,-2.3264,-2.15120,-1.9760,-2.56,-16.917808,zhou xuanyu,None,
1,16656,Zhong Chuhan,12.26,None,None,None,None,Triple Jump,None,None,...,12.26,12.26,0.8136,0.98880,1.1640,0.58,9.965753,chuhan zhong,2005-10-18,Tier 1
2,16364,Zheng Justin De,10.47m,National Junior College,14,12,U15,Triple Jump,0,None,...,10.47,10.47,-3.5342,-3.31985,-3.1055,-3.82,-21.731980,zheng justin de,2011-02-21,
3,16497,"Zhao, Daniel",11.67m,Hwa Chong Institution,14,3,U15,Triple Jump,0,None,...,11.67,11.67,-2.3342,-2.11985,-1.9055,-2.62,-13.334500,zhao daniel,2011-07-25,
4,18293,Zhao Daniel,11.98,-,,1,,Triple Jump,,nan,...,11.98,11.98,-2.0242,-1.80985,-1.5955,-2.31,-11.165150,daniel zhao,2011-07-25,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12505,7003,"., Shaik Isa",15.49,Club ZOOM,8,32,U9,Dash,80,,...,,NaN,NaN,NaN,NaN,NaN,NaN,shaik isa,2016-11-21,
12506,6660,"., Nur Elena",16.42,UNA,6,1,U7,Dash,80,,...,,NaN,NaN,NaN,NaN,NaN,NaN,nur elena,2018-12-04,
12507,13258,"., Dharkshitha",08:35.6,Cedar Girls Secondary School,15,1,U18,Race Walk,1500,None,...,,NaN,NaN,NaN,NaN,NaN,NaN,dharkshitha,2010-06-04,
12508,12881,"., Cayden",06:05.1,North Vista,12,54,Open,Mile Run,1,,...,,NaN,NaN,NaN,NaN,NaN,NaN,cayden,2010-01-01,


In [650]:
# Choose rows with mapped event

final_df=final_df[(final_df['MAPPED_EVENT']!=' ') & (final_df['TIER']!=' ')]

In [652]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/17th SEA Youth/')


final_df.to_csv('U18_tiered_performers_oct29.csv', encoding='utf-8')

In [653]:
final_df

,index,NAME,RESULT_x,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT_x,DISTANCE,EVENT_CLASS,...,RESULT_CONV,RESULT_BEST,Delta2,Delta3.5,Delta5,Delta_Benchmark,PERF_SCALAR,clean_name,DOB_parsed,TIER
1,16656,Zhong Chuhan,12.26,None,None,None,None,Triple Jump,None,None,...,12.26,12.26,0.8136,0.98880,1.1640,0.58,9.965753,chuhan zhong,2005-10-18,Tier 1
55,16332,Tang Kai Sheng Cayman,14.17m,VICTORIA SCHOOL,16,1,U18,Triple Jump,0,None,...,14.17,14.17,0.1658,0.38015,0.5945,-0.12,4.160252,tang cayman,2009-07-15,Tier 2
59,16442,"Tan, Tse Teng",11.77m,Hwa Chong Alumni Association,12,2,Open,Triple Jump,0,None,...,11.77,11.77,0.3236,0.49880,0.6740,0.09,5.770548,tan tse teng,2002-11-28,Tier 1
66,613,"Tan, Jurnus",11.30m,Wings Athletics Club,12,1,Open,Triple Jump,0.0,None,...,11.3,11.30,-0.1464,0.02880,0.2040,-0.38,1.746575,tan jurnus,2004-12-05,Tier 3
76,16654,Tan Tse Teng,11.59,None,None,None,None,Triple Jump,None,None,...,11.59,11.59,0.1436,0.31880,0.4940,-0.09,4.229452,tse teng tan,None,Tier 2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9331,2304,Alexa Tan Liyi,00:12.98,MGS,16,3.0,B,100m,None,None,...,12.98,12.98,-0.2198,-0.03215,0.1555,-0.47,1.243006,alexa tan liyi,None,Tier 4
9337,1658,Aidan Michael Soh Zi Ren,11.16,INDIVIDUAL,18,None,None,100m,None,None,...,11.16,11.16,-0.2052,-0.04410,0.1170,-0.42,1.089385,aidan michael soh,2007-10-29,Tier 4
9355,488,"Abisha, Ivan",11.23,Oldham Athletics,12,10,Open,Dash,100.0,None,...,11.23,11.23,-0.2752,-0.11410,0.0470,-0.49,0.437616,abisha ivan,2000-12-24,Tier 4
9358,9859,"Abdul Latiff, Erza Irdina",13.03,Club ZOOM,12,1,Open,Dash,100.0,,...,13.03,13.03,-0.2698,-0.08215,0.1055,-0.52,0.843325,abdul latiff erza irdina,2002-01-01,Tier 4


In [654]:
#os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/Asian Youth/')


#final_selection.to_csv('AY_final_selection.csv', encoding='utf-8')

In [655]:
# Get list of names with no DOB

mask = ((final_df['DOB']=='None')|(final_df['DOB']=='nan'))

#mask = ((final_df['DOB'].isnull()))
#mask = final_df['DOB_new']=='NaT'

missing_names = final_df.loc[mask]

name_list = missing_names['NAME'].str.casefold().to_list()


In [656]:
len(name_list)

171

In [657]:
# Names with no DOB

missing_names

,index,NAME,RESULT_x,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT_x,DISTANCE,EVENT_CLASS,...,RESULT_CONV,RESULT_BEST,Delta2,Delta3.5,Delta5,Delta_Benchmark,PERF_SCALAR,clean_name,DOB_parsed,TIER
76,16654,Tan Tse Teng,11.59,None,None,None,None,Triple Jump,None,None,...,11.59,11.59,0.1436,0.31880,0.4940,-0.09,4.229452,tse teng tan,None,Tier 2
79,16613,Tan Shou Ri Yei (Chen Shouyi),14.31,RI,18.5,1.0,A,Triple Jump,None,None,...,14.31,14.31,0.3058,0.52015,0.7345,0.02,5.139958,tan shou yi rei chen shouyi,None,Tier 1
103,16294,"Saw, Xiang Yu",13.92m,Singapore Institute of Managem,12,2,Open,Triple Jump,0,None,...,13.92,13.92,-0.0842,0.13015,0.3445,-0.37,2.410777,saw xiang yu,None,Tier 3
204,16614,Lau Jia Hern,13.63,RI,18.5,3.0,A,Triple Jump,None,None,...,13.63,13.63,-0.3742,-0.15985,0.0545,-0.66,0.381386,lau jia hern,None,Tier 4
289,16300,Chew Yiu Tse Jade,11.13m,National University Singapore,12,2,Open,Triple Jump,0,None,...,11.13,11.13,-0.3164,-0.14120,0.0340,-0.55,0.291096,chew jade,None,Tier 4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9116,2192,Chloe Sarahena Tan Hui En,00:12.85,EJC,18.5,6.0,A,100m,None,None,...,12.85,12.85,-0.0898,0.09785,0.2855,-0.34,2.282174,chloe sarahena tan hui en,None,Tier 3
9117,2288,Chloe Chee En-Ya,00:12.44,ACS(I),18.5,4.0,A,100m,None,None,...,12.44,12.44,0.3202,0.50785,0.6955,0.07,5.559552,chloe chee enya,None,Tier 1
9221,1682,Bryan Ng,10.93,None,None,2,None,100m,None,None,...,10.93,10.93,0.0248,0.18590,0.3470,-0.19,3.230912,bryan ng,None,Tier 2
9227,2319,Brayden Chan Wei Jie,00:10.95,RI,18.5,3.0,A,100m,None,None,...,10.95,10.95,0.0048,0.16590,0.3270,-0.21,3.044693,brayden chan wei jie,None,Tier 2


In [658]:
final_df["DOB"] = pd.to_datetime(final_df["DOB"], errors="coerce", dayfirst=True)


/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_16781/2569036124.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  final_df["DOB"] = pd.to_datetime(final_df["DOB"], errors="coerce", dayfirst=True)
/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_16781/2569036124.py:1: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  final_df["DOB"] = pd.to_datetime(final_df["DOB"], errors="coerce", dayfirst=True)
/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_16781/2569036124.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a D

In [659]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/17th SEA Youth/')

final_df.to_csv('before_name_lookup.csv', encoding='utf-8')

In [660]:
# Create a list of name components to seach for permutations

#final_df['name_to_list'] = final_df['clean_name'].str.split(' ')
final_df.loc[:, 'name_to_list'] = final_df['clean_name'].str.split(' ')


final_df


/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_16781/2896884656.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df.loc[:, 'name_to_list'] = final_df['clean_name'].str.split(' ')


,index,NAME,RESULT_x,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT_x,DISTANCE,EVENT_CLASS,...,RESULT_BEST,Delta2,Delta3.5,Delta5,Delta_Benchmark,PERF_SCALAR,clean_name,DOB_parsed,TIER,name_to_list
1,16656,Zhong Chuhan,12.26,None,None,None,None,Triple Jump,None,None,...,12.26,0.8136,0.98880,1.1640,0.58,9.965753,chuhan zhong,2005-10-18,Tier 1,"[chuhan, zhong]"
55,16332,Tang Kai Sheng Cayman,14.17m,VICTORIA SCHOOL,16,1,U18,Triple Jump,0,None,...,14.17,0.1658,0.38015,0.5945,-0.12,4.160252,tang cayman,2009-07-15,Tier 2,"[tang, cayman]"
59,16442,"Tan, Tse Teng",11.77m,Hwa Chong Alumni Association,12,2,Open,Triple Jump,0,None,...,11.77,0.3236,0.49880,0.6740,0.09,5.770548,tan tse teng,2002-11-28,Tier 1,"[tan, tse, teng]"
66,613,"Tan, Jurnus",11.30m,Wings Athletics Club,12,1,Open,Triple Jump,0.0,None,...,11.30,-0.1464,0.02880,0.2040,-0.38,1.746575,tan jurnus,2004-12-05,Tier 3,"[tan, jurnus]"
76,16654,Tan Tse Teng,11.59,None,None,None,None,Triple Jump,None,None,...,11.59,0.1436,0.31880,0.4940,-0.09,4.229452,tse teng tan,None,Tier 2,"[tse, teng, tan]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9331,2304,Alexa Tan Liyi,00:12.98,MGS,16,3.0,B,100m,None,None,...,12.98,-0.2198,-0.03215,0.1555,-0.47,1.243006,alexa tan liyi,None,Tier 4,"[alexa, tan, liyi]"
9337,1658,Aidan Michael Soh Zi Ren,11.16,INDIVIDUAL,18,None,None,100m,None,None,...,11.16,-0.2052,-0.04410,0.1170,-0.42,1.089385,aidan michael soh,2007-10-29,Tier 4,"[aidan, michael, soh]"
9355,488,"Abisha, Ivan",11.23,Oldham Athletics,12,10,Open,Dash,100.0,None,...,11.23,-0.2752,-0.11410,0.0470,-0.49,0.437616,abisha ivan,2000-12-24,Tier 4,"[abisha, ivan]"
9358,9859,"Abdul Latiff, Erza Irdina",13.03,Club ZOOM,12,1,Open,Dash,100.0,,...,13.03,-0.2698,-0.08215,0.1055,-0.52,0.843325,abdul latiff erza irdina,2002-01-01,Tier 4,"[abdul, latiff, erza, irdina]"


In [661]:
# Lookup DOB from name

final_df['dob_list'] = None  


for index, row in final_df.iterrows():
         
   # if str(row['DOB_new']) == 'NaT':
   # if ((final_df['DOB']=='None') or (final_df['DOB']=='nan')):
            
    name_components = row['name_to_list']
        
    permutations_list = []
    dob_list=[]
    string=''
    
    
    for p in permutations(name_components):
        
        string = " ".join(p)
        
        permutations_list.append(string)

        dob = dictionary_dob_clean_name.get(string)
        
        if dob is not None:
            dob_list.append(dob)

    
  #  null.at[index, 'permutations'] = permutations_list
   # athletes.loc[index, 'dob_list'] = dob_list
    final_df.at[index, 'dob_list'] = [dob_list] if isinstance(dob_list, list) else dob_list




/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_16781/4220560926.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df['dob_list'] = None


In [663]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/17th SEA Youth/')

final_df.to_csv('post_dob_list_oct29.csv', encoding='utf-8')

In [664]:
# Find the missing DOBs from all the possible name permutations
# Saves permuations list and has more robust handling of missing name components

from itertools import permutations

# Make sure the target columns exist
#final_df['permutations'] = None
final_df['dob_list'] = None

for index, row in final_df.iterrows():
    name_components = row['name_to_list']
    
    # If the row has no valid components, skip
    if not isinstance(name_components, (list, tuple)) or len(name_components) == 0:
        athletes.at[index, 'permutations'] = []
        athletes.at[index, 'dob_list'] = []
        continue
    
    permutations_list = []
    dob_list = []
    
    for p in permutations(name_components):
        string = " ".join(p)
        permutations_list.append(string)

        dob = dictionary_dob_clean_name.get(string)
        if dob is not None:
            dob_list.append(dob)

    # Save lists directly into cells (as Python objects)
  #  final_df.at[index, 'permutations'] = permutations_list
    final_df.at[index, 'dob_list'] = dob_list


/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_16781/3455906722.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df['dob_list'] = None


In [665]:
final_df

,index,NAME,RESULT_x,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT_x,DISTANCE,EVENT_CLASS,...,Delta2,Delta3.5,Delta5,Delta_Benchmark,PERF_SCALAR,clean_name,DOB_parsed,TIER,name_to_list,dob_list
1,16656,Zhong Chuhan,12.26,None,None,None,None,Triple Jump,None,None,...,0.8136,0.98880,1.1640,0.58,9.965753,chuhan zhong,2005-10-18,Tier 1,"[chuhan, zhong]","[2005-10-18, 2005-10-18]"
55,16332,Tang Kai Sheng Cayman,14.17m,VICTORIA SCHOOL,16,1,U18,Triple Jump,0,None,...,0.1658,0.38015,0.5945,-0.12,4.160252,tang cayman,2009-07-15,Tier 2,"[tang, cayman]",[2009-07-15]
59,16442,"Tan, Tse Teng",11.77m,Hwa Chong Alumni Association,12,2,Open,Triple Jump,0,None,...,0.3236,0.49880,0.6740,0.09,5.770548,tan tse teng,2002-11-28,Tier 1,"[tan, tse, teng]","[2002-11-28, 2002-11-28]"
66,613,"Tan, Jurnus",11.30m,Wings Athletics Club,12,1,Open,Triple Jump,0.0,None,...,-0.1464,0.02880,0.2040,-0.38,1.746575,tan jurnus,2004-12-05,Tier 3,"[tan, jurnus]",[2004-12-05]
76,16654,Tan Tse Teng,11.59,None,None,None,None,Triple Jump,None,None,...,0.1436,0.31880,0.4940,-0.09,4.229452,tse teng tan,None,Tier 2,"[tse, teng, tan]","[2002-11-28, 2002-11-28]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9331,2304,Alexa Tan Liyi,00:12.98,MGS,16,3.0,B,100m,None,None,...,-0.2198,-0.03215,0.1555,-0.47,1.243006,alexa tan liyi,None,Tier 4,"[alexa, tan, liyi]",[]
9337,1658,Aidan Michael Soh Zi Ren,11.16,INDIVIDUAL,18,None,None,100m,None,None,...,-0.2052,-0.04410,0.1170,-0.42,1.089385,aidan michael soh,2007-10-29,Tier 4,"[aidan, michael, soh]",[2007-10-29]
9355,488,"Abisha, Ivan",11.23,Oldham Athletics,12,10,Open,Dash,100.0,None,...,-0.2752,-0.11410,0.0470,-0.49,0.437616,abisha ivan,2000-12-24,Tier 4,"[abisha, ivan]",[2000-12-24]
9358,9859,"Abdul Latiff, Erza Irdina",13.03,Club ZOOM,12,1,Open,Dash,100.0,,...,-0.2698,-0.08215,0.1055,-0.52,0.843325,abdul latiff erza irdina,2002-01-01,Tier 4,"[abdul, latiff, erza, irdina]",[2002-01-01]


In [666]:
# Create a new column with the first element of dob_list (or NaN if empty)

final_df['dob_first'] = final_df['dob_list'].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None)


/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_16781/3894324010.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df['dob_first'] = final_df['dob_list'].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None)


In [667]:
final_df

,index,NAME,RESULT_x,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT_x,DISTANCE,EVENT_CLASS,...,Delta3.5,Delta5,Delta_Benchmark,PERF_SCALAR,clean_name,DOB_parsed,TIER,name_to_list,dob_list,dob_first
1,16656,Zhong Chuhan,12.26,None,None,None,None,Triple Jump,None,None,...,0.98880,1.1640,0.58,9.965753,chuhan zhong,2005-10-18,Tier 1,"[chuhan, zhong]","[2005-10-18, 2005-10-18]",2005-10-18
55,16332,Tang Kai Sheng Cayman,14.17m,VICTORIA SCHOOL,16,1,U18,Triple Jump,0,None,...,0.38015,0.5945,-0.12,4.160252,tang cayman,2009-07-15,Tier 2,"[tang, cayman]",[2009-07-15],2009-07-15
59,16442,"Tan, Tse Teng",11.77m,Hwa Chong Alumni Association,12,2,Open,Triple Jump,0,None,...,0.49880,0.6740,0.09,5.770548,tan tse teng,2002-11-28,Tier 1,"[tan, tse, teng]","[2002-11-28, 2002-11-28]",2002-11-28
66,613,"Tan, Jurnus",11.30m,Wings Athletics Club,12,1,Open,Triple Jump,0.0,None,...,0.02880,0.2040,-0.38,1.746575,tan jurnus,2004-12-05,Tier 3,"[tan, jurnus]",[2004-12-05],2004-12-05
76,16654,Tan Tse Teng,11.59,None,None,None,None,Triple Jump,None,None,...,0.31880,0.4940,-0.09,4.229452,tse teng tan,None,Tier 2,"[tse, teng, tan]","[2002-11-28, 2002-11-28]",2002-11-28
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9331,2304,Alexa Tan Liyi,00:12.98,MGS,16,3.0,B,100m,None,None,...,-0.03215,0.1555,-0.47,1.243006,alexa tan liyi,None,Tier 4,"[alexa, tan, liyi]",[],None
9337,1658,Aidan Michael Soh Zi Ren,11.16,INDIVIDUAL,18,None,None,100m,None,None,...,-0.04410,0.1170,-0.42,1.089385,aidan michael soh,2007-10-29,Tier 4,"[aidan, michael, soh]",[2007-10-29],2007-10-29
9355,488,"Abisha, Ivan",11.23,Oldham Athletics,12,10,Open,Dash,100.0,None,...,-0.11410,0.0470,-0.49,0.437616,abisha ivan,2000-12-24,Tier 4,"[abisha, ivan]",[2000-12-24],2000-12-24
9358,9859,"Abdul Latiff, Erza Irdina",13.03,Club ZOOM,12,1,Open,Dash,100.0,,...,-0.08215,0.1055,-0.52,0.843325,abdul latiff erza irdina,2002-01-01,Tier 4,"[abdul, latiff, erza, irdina]",[2002-01-01],2002-01-01


In [668]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/17th SEA Youth/')

final_df.to_csv('check.csv', encoding='utf-8')

In [669]:
# Filter athletes born between 1 Jan 2008 to 31 Dec 2009

# First, convert to datetime
final_df['dob_first'] = pd.to_datetime(final_df['dob_first'], errors='coerce')

# Then remove timezone if it exists
if final_df['dob_first'].dt.tz is not None:
    
    final_df['dob_first'] = final_df['dob_first'].dt.tz_localize(None)

start = datetime.datetime(2008, 1, 1)

end = datetime.datetime(2009, 12, 31)

start_date = np.datetime64(start)
end_date = np.datetime64(end)


mask = ((final_df['dob_first'] >= start_date) & (final_df['dob_first'] <= end_date)|(final_df['dob_first'].isna()))
athletes_selected = final_df.loc[mask]



/var/folders/q5/yf8g5p896_b94gkbhqcjx3t40000gn/T/ipykernel_16781/966973162.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df['dob_first'] = pd.to_datetime(final_df['dob_first'], errors='coerce')


In [670]:
athletes_selected

,index,NAME,RESULT_x,TEAM,AGE,COMPETITION_RANK,DIVISION,EVENT_x,DISTANCE,EVENT_CLASS,...,Delta3.5,Delta5,Delta_Benchmark,PERF_SCALAR,clean_name,DOB_parsed,TIER,name_to_list,dob_list,dob_first
55,16332,Tang Kai Sheng Cayman,14.17m,VICTORIA SCHOOL,16,1,U18,Triple Jump,0,None,...,0.38015,0.5945,-0.12,4.160252,tang cayman,2009-07-15,Tier 2,"[tang, cayman]",[2009-07-15],2009-07-15
78,16419,Tan Shou Yi Rei,14.99m,Raffles Institution JC,17,1,U20,Triple Jump,0,None,...,1.20015,1.4145,0.70,9.898530,tan rei,2008-12-05,Tier 1,"[tan, rei]","[2008-12-05, 2008-04-12]",2008-12-05
79,16613,Tan Shou Ri Yei (Chen Shouyi),14.31,RI,18.5,1.0,A,Triple Jump,None,None,...,0.52015,0.7345,0.02,5.139958,tan shou yi rei chen shouyi,None,Tier 1,"[tan, shou, yi, rei, chen, shouyi]",[],NaT
80,16453,Tan Shou Ri Yei,13.76,None,None,1,None,Triple Jump,None,None,...,-0.02985,0.1845,-0.53,1.291113,tan shou yi rei,2007-10-29,Tier 4,"[tan, shou, yi, rei]",[2008-10-29],2008-10-29
204,16614,Lau Jia Hern,13.63,RI,18.5,3.0,A,Triple Jump,None,None,...,-0.15985,0.0545,-0.66,0.381386,lau jia hern,None,Tier 4,"[lau, jia, hern]","[2008-09-03, 2008-03-09]",2008-09-03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9104,2325,Chong Lynn Xi,00:13.13,CGS,16,5.0,B,100m,None,None,...,-0.18215,0.0055,-0.62,0.043965,chong lynn xi,None,Tier 4,"[chong, lynn, xi]",[],NaT
9116,2192,Chloe Sarahena Tan Hui En,00:12.85,EJC,18.5,6.0,A,100m,None,None,...,0.09785,0.2855,-0.34,2.282174,chloe sarahena tan hui en,None,Tier 3,"[chloe, sarahena, tan, hui, en]",[],NaT
9117,2288,Chloe Chee En-Ya,00:12.44,ACS(I),18.5,4.0,A,100m,None,None,...,0.50785,0.6955,0.07,5.559552,chloe chee enya,None,Tier 1,"[chloe, chee, enya]","[2008-03-28, 2008-01-01, 2008-03-28]",2008-03-28
9163,9862,"Chee, Chloe En-Ya",12.69,Oldham Athletics,17,1,U18,Dash,100.0,,...,0.25785,0.4455,-0.18,3.561151,chee chloe enya,2008-01-01,Tier 2,"[chee, chloe, enya]","[2008-01-01, 2008-03-28, 2008-03-28]",2008-01-01


In [671]:
os.chdir('/Users/veesheenyuen/Desktop/DataScience/SAA/17th SEA Youth/')

athletes_selected.to_csv('final_selection_U18_oct29.csv', encoding='utf-8')